In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# AI-Powered Job Recommendation & Skill Gap Analysis
### End-to-end notebook | Indian Job Market Dataset 2025 + O*NET reference

**Scope:** all IT job families present in the dataset (not only AI/ML).

**What this notebook builds**
1. Clean and inspect job listings; preserve the raw dataset.
2. Identify likely IT listings with a transparent, editable title taxonomy.
3. Normalize skills and create a searchable job corpus.
4. Parse a resume PDF/DOCX or accept pasted resume text.
5. Rank jobs using semantic-ish TF-IDF text similarity + skill overlap + experience fit.
6. Explain each recommendation and show missing / detected skills.
7. Generate a prioritized learning roadmap using job-market demand and an explicit prerequisite graph.
8. Run sanity checks and create a small manual validation sheet.

**Important limitations**
- This is a recommendation prototype, not a hiring/eligibility decision system.
- Resume parsing and skills extraction are imperfect; “not detected” does not prove a person lacks a skill.
- O*NET is a US occupational reference. It is supplementary and does not replace the Indian job dataset.
- Similarity scores are ranking signals, not probabilities of getting hired.
- Do not publish personal resumes or contact details in a public repository.

## 0. Runtime and data setup

Recommended: Google Colab with a standard CPU runtime. GPU is **not required**.

Upload these two files to the Colab session using the left **Files** panel, or place them in Google Drive:
- `indian-job-market-dataset-2025.xlsx`
- `onet-20261025T130155Z-1-001.zip` (contains the seven O*NET CSV files)

If you prefer Drive, edit `DRIVE_BASE` below. The notebook searches common locations and asks for upload only if it cannot find the files.

In [ ]:
from pathlib import Path
import os, re, json, zipfile, html, warnings, textwrap
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

SEED = 42
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 100)

# Optional: set this to your Drive folder, e.g.
# DRIVE_BASE = Path("/content/drive/MyDrive/Job recommendation nd skill gap analysis")
DRIVE_BASE = Path("/content/drive/MyDrive/Job recommendation nd skill gap analysis")

**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
def locate_file(filename):
    candidates = [
        Path("/content") / filename,
        Path("/content/drive/MyDrive") / filename,
        Path("/mnt/data") / filename,
    ]
    if DRIVE_BASE:
        candidates.insert(0, Path(DRIVE_BASE) / filename)
        candidates.insert(0, Path(DRIVE_BASE) / "raw" / filename)
    for p in candidates:
        if p.exists():
            return p
    return None

RAW_PATH = locate_file("indian-job-market-dataset-2025.xlsx")
ONET_ZIP = locate_file("onet-20261025T130155Z-1-001.zip")

print("Excel:", RAW_PATH)
print("O*NET ZIP:", ONET_ZIP)

Excel: /content/drive/MyDrive/Job recommendation nd skill gap analysis/raw/indian-job-market-dataset-2025.xlsx
O*NET ZIP: None


**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
if RAW_PATH is None:
    try:
        from google.colab import files
        print("Upload the missing Excel and/or O*NET ZIP file(s).")
        uploaded = files.upload()
        if RAW_PATH is None:
            p = Path("/content/indian-job-market-dataset-2025.xlsx")
            if p.exists(): RAW_PATH = p
        if ONET_ZIP is None:
            p = Path("/content/onet-20261025T130155Z-1-001.zip")
            if p.exists(): ONET_ZIP = p
    except Exception:
        pass

if RAW_PATH is None:
    raise FileNotFoundError("Could not find the Excel dataset. Upload indian-job-market-dataset-2025.xlsx.")
if ONET_ZIP is None:
    print("O*NET ZIP not found. The core project can run without O*NET; occupation enrichment will be skipped.")

O*NET ZIP not found. The core project can run without O*NET; occupation enrichment will be skipped.


In [ ]:
onet = {}
ONET_FOLDER = Path(DRIVE_BASE) / "onet"
for filename in ["occupation_data.csv", "job_titles.csv", "software_skills.csv",
                 "career_interest_types.csv", "knowledge.csv",
                 "related_occupations.csv", "job_zones.csv"]:
    onet[filename] = pd.read_csv(ONET_FOLDER / filename)
print({k: v.shape for k, v in onet.items()})

{'occupation_data.csv': (1016, 3), 'job_titles.csv': (54269, 5), 'software_skills.csv': (31821, 7), 'career_interest_types.csv': (8307, 9), 'knowledge.csv': (60060, 15), 'related_occupations.csv': (18460, 6), 'job_zones.csv': (923, 5)}


## 1. Load the source data and inspect its schema

In [ ]:
df_raw = pd.read_excel(RAW_PATH)
print("Raw shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
display(df_raw.head(5))
display(df_raw.isna().mean().sort_values(ascending=False).rename("missing_fraction").to_frame().head(15))

Raw shape: (97929, 17)
Columns: ['title', 'jobId', 'currency', 'jobUploaded', 'companyName', 'tagsAndSkills', 'experience', 'salary', 'location', 'companyId', 'ReviewsCount', 'AggregateRating', 'jobDescription', 'minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience']


,title,jobId,currency,jobUploaded,companyName,tagsAndSkills,experience,salary,location,companyId,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience
0,Sr. HR Recruiter (NON IT),270925008041,INR,6 Days Ago,Orion,"Communication,Manpower,Staffing,Convincing Power,Hiring,Recruitment,SR,Communication skills",2-4 Yrs,2-4 Lacs PA,Kolkata(Chinar Park),645563,NaN,NaN,Preferred candidate profile . .,200000,400000,2,4
1,Fire And Safety Officer,270925007584,INR,6 Days Ago,"Apollo Hospitals International Limited, Ahmedabad","Safety Officer Activities,Fire Protection,Fire Safety,Fire Engineering,Fire Prevention,Safety Ma...",6-11 Yrs,3-5 Lacs PA,"Gandhinagar, Ahmedabad",14072,5162,4.0,"Ensure active Fire Protection System,such as Fire Hydrant system,Fire extinguishers,Fire Alarm s...",300000,500000,6,11
2,Opening For Performance Marketing - Chennai,270925007492,INR,6 Days Ago,TVS Credit Services Ltd,"Performance Marketing,User Acquisition,growth marketing,Paid Marketing,Acquisition,Performance,U...",12-18 Yrs,Not disclosed,Chennai,1324750,2892,4.2,MBA Marketing (preferred Tier II or III B- School). <br><br>Experience <br><br> 12+ years exp in...,0,0,12,18
3,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive
4,Senior Group Product Manager - CNS Therapy,270925007430,INR,6 Days Ago,Cadila Pharmaceuticals,"Product Marketing,CNS,Product Management,Nephrology,Group Product Management,Brand Marketing,Neu...",5-10 Yrs,8-18 Lacs PA,Ahmedabad,14957,2134,3.4,Principal Tasks & Responsibilities : (Please write all the major jobs that the employee is requi...,800000,1800000,5,10


,missing_fraction
AggregateRating,0.359965
ReviewsCount,0.359965
experience,0.021495
minimumSalary,0.005831
tagsAndSkills,0.005831
maximumExperience,0.005831
maximumSalary,0.005831
minimumExperience,0.005831
companyName,0.000041
title,0.000000


## 2. Clean job listings
We keep the original columns, add cleaned columns, and document assumptions. Salary columns are treated as INR per annum only when the source values and `currency` support that assumption; the source's salary text is preserved.

In [ ]:
required_cols = ["title", "jobId", "companyName", "tagsAndSkills", "jobDescription", "location"]
missing_cols = [c for c in required_cols if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Dataset schema changed; missing columns: {missing_cols}")

PHONE_RE = re.compile(r"(?<!\d)(?:\+?91[-\s]?)?[6-9]\d{9}(?!\d)")
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.I)
HTML_RE = re.compile(r"<[^>]+>")
CITY_FIX = {"bangalore":"Bengaluru", "gurgaon":"Gurugram", "bombay":"Mumbai",
            "madras":"Chennai", "calcutta":"Kolkata", "new delhi":"Delhi"}

def clean_text(value):
    if pd.isna(value):
        return ""
    s = html.unescape(str(value))
    s = HTML_RE.sub(" ", s)
    s = URL_RE.sub(" ", s)
    s = EMAIL_RE.sub(" ", s)
    s = PHONE_RE.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
def normalize_text(value):
    s = clean_text(value).lower()
    s = re.sub(r"[^a-z0-9+#./ -]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def parse_skills(value):
    if pd.isna(value):
        return []
    out, seen = [], set()
    for item in str(value).split(","):
        skill = re.sub(r"\s+", " ", item.lower()).strip(" .;:/")
        if skill and skill not in seen:
            seen.add(skill)
            out.append(skill)
    return out

def normalize_city(value):
    if pd.isna(value):
        return "Unknown"
    first = re.split(r"[,(/]", str(value))[0].strip()
    return CITY_FIX.get(first.lower(), first.title()) if first else "Unknown"

df = df_raw.copy()

**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
df["jobId"] = df["jobId"].astype("string")

# Vectorized cleaning is substantially faster on ~100k listings than calling regexes row by row.
def clean_series(series):
    s = series.fillna("").astype(str).map(html.unescape)
    s = s.str.replace(HTML_RE, " ", regex=True)
    s = s.str.replace(URL_RE, " ", regex=True)
    s = s.str.replace(EMAIL_RE, " ", regex=True)
    s = s.str.replace(PHONE_RE, " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

df["title_clean"] = clean_series(df["title"])
df["title_norm"] = (df["title_clean"].str.lower()
    .str.replace(r"[^a-z0-9+#./ -]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True).str.strip())
df["description_clean"] = clean_series(df["jobDescription"])
df["skills_list"] = df["tagsAndSkills"].map(parse_skills)
df["skills_text"] = df["skills_list"].map(lambda xs: " ".join(xs))
df["city"] = df["location"].map(normalize_city)
df["experience_min"] = pd.to_numeric(df.get("minimumExperience", pd.Series(index=df.index, dtype=float)), errors="coerce")
df["experience_max"] = pd.to_numeric(df.get("maximumExperience", pd.Series(index=df.index, dtype=float)), errors="coerce")

before = len(df)
df = df.drop_duplicates(subset=["jobId"], keep="first")

**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
after_id_dedup = len(df)
df["_content_key"] = (
    df["title_norm"].fillna("") + "|" +
    df["companyName"].fillna("").astype(str).str.lower().str.strip() + "|" +
    df["location"].fillna("").astype(str).str.lower().str.strip() + "|" +
    df["description_clean"].fillna("").astype(str).str.lower().str.replace(r"[^a-z0-9+#./ -]", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
)
df = df.drop_duplicates(subset=["_content_key"], keep="first").drop(columns="_content_key")
print(f"Rows: {before:,} -> after jobId dedupe {after_id_dedup:,} -> after exact-content dedupe {len(df):,}")
print("Blank title fraction:", round((df["title_norm"] == "").mean(), 4))
print("Empty skills fraction:", round(df["skills_list"].map(len).eq(0).mean(), 4))
display(df[["title", "companyName", "location", "experience", "salary", "skills_list"]].head(8))

Rows: 97,929 -> after jobId dedupe 97,679 -> after exact-content dedupe 93,729
Blank title fraction: 0.0
Empty skills fraction: 0.0059


,title,companyName,location,experience,salary,skills_list
0,Sr. HR Recruiter (NON IT),Orion,Kolkata(Chinar Park),2-4 Yrs,2-4 Lacs PA,"[communication, manpower, staffing, convincing power, hiring, recruitment, sr, communication ski..."
1,Fire And Safety Officer,"Apollo Hospitals International Limited, Ahmedabad","Gandhinagar, Ahmedabad",6-11 Yrs,3-5 Lacs PA,"[safety officer activities, fire protection, fire safety, fire engineering, fire prevention, saf..."
2,Opening For Performance Marketing - Chennai,TVS Credit Services Ltd,Chennai,12-18 Yrs,Not disclosed,"[performance marketing, user acquisition, growth marketing, paid marketing, acquisition, perform..."
3,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,Medical Billing Executive,[medical billing executive]
4,Senior Group Product Manager - CNS Therapy,Cadila Pharmaceuticals,Ahmedabad,5-10 Yrs,8-18 Lacs PA,"[product marketing, cns, product management, nephrology, group product management, brand marketi..."
5,Senior Accountant,Vedaant Techno Art,"Noida, Greater Noida",10-15 Yrs,3.5-6 Lacs PA,"[tds filing, itr, gst filing, balance sheet finalisation, tax audit, return filing, accounts fin..."
6,Maintenance Technician,Avalon Technologies,"Sriperumbudur, Chennai, Kanchipuram",3-5 Yrs,Not disclosed,"[booth, conveyor, special process, utility maintenance, powder coating, preventive maintenance, ..."
7,Officer/Engineer Technical,Balkrishna Industries (BKT),Aurangabad(Waluj),2-3 Yrs,Not disclosed,"[m.s office, sap, computer skills, ms office, technical, computer knowledge, computer, office]"


In [ ]:
df["city"] = df["city"].str.replace(r"^Hybrid\s*-\s*", "", regex=True)

## 3. Exploratory data analysis

In [ ]:
print("Rows:", f"{len(df):,}")
print("Unique companies:", df["companyName"].nunique(dropna=True))
print("Unique cities:", df["city"].nunique())
print("Listings with at least one parsed skill:", f"{df['skills_list'].map(len).gt(0).mean():.1%}")
display(df["city"].value_counts().head(15).rename_axis("city").to_frame("listings"))
display(df["experience"].fillna("Not specified").value_counts().head(12).rename_axis("experience").to_frame("listings"))
display(df["skills_list"].explode().value_counts().head(30).rename_axis("skill").to_frame("listing_skill_mentions"))

Rows: 93,729
Unique companies: 18668
Unique cities: 1278
Listings with at least one parsed skill: 99.4%


,listings
city,
Bengaluru,18226
Hyderabad,11931
Pune,8965
Mumbai,6285
Chennai,6260
Gurugram,5187
Noida,4918
Ahmedabad,2399
Kolkata,1914


,listings
experience,
5-10 Yrs,6295
3-8 Yrs,5400
1-3 Yrs,3620
2-5 Yrs,3577
2-7 Yrs,3553
1-6 Yrs,3536
0-5 Yrs,3162
3-5 Yrs,3007
4-9 Yrs,2806


,listing_skill_mentions
skill,
sales,8916
python,5431
project management,5181
management,4802
sap,4590
customer service,4564
java,3781
sql,3747
css,3627


## 4. Classify job listings into IT families

This is a **transparent rule-based baseline** using job titles. All IT families are included. Rules are intentionally editable and the final `Other / Non-IT or Unclear` bucket must be manually reviewed before reporting classification quality.

Avoid using one keyword such as `python`, `spark`, `cloud`, or `developer` alone as a definitive role label: those words can appear in unrelated or mixed roles.

In [ ]:
FAMILY_RULES = [
    ("Cybersecurity", r"\bcyber ?security\b|\binformation security\b|\binfosec\b|\bsecurity analyst\b|\bsecurity engineer\b|penetration test|\bvapt\b|\bsoc analyst\b|\biam engineer\b"),
    ("Data Scientist / ML-AI", r"data scientist|machine learning|\bartificial intelligence\b|\bai engineer\b|\bml engineer\b|deep learning|computer vision|\bnlp engineer\b|generative ai|\bllm engineer\b|\bdata science\b"),
    ("Data Engineer", r"data engineer|data engineering|etl developer|etl engineer|data pipeline|data platform engineer|big data engineer|data warehouse engineer"),
    ("Data / BI Analyst", r"data analyst|business intelligence|\bbi analyst\b|power bi|tableau developer|reporting analyst|analytics analyst|business data analyst"),
    ("Database Admin", r"database administrator|\bdba\b|database engineer|database developer|oracle database admin|sql database administrator"),
    ("DevOps / Cloud", r"devops|site reliability|\bsre\b|cloud engineer|cloud architect|cloud administrator|platform engineer|kubernetes engineer|\baws engineer\b|\bazure engineer\b|\bgcp engineer\b|infrastructure engineer"),
    ("QA / Testing", r"quality assurance|\bqa engineer\b|\bqa analyst\b|software tester|test engineer|testing engineer|automation test|\bsdet\b"),
    ("Mobile Developer", r"android developer|ios developer|mobile app developer|mobile developer|flutter developer|react native developer|swift developer|kotlin developer"),
    ("Frontend / Web", r"frontend|front end developer|web developer|ui developer|react developer|angular developer|vue developer|wordpress developer|php developer"),
    ("ERP / CRM Consultant", r"\bsap\b|salesforce|servicenow|dynamics 365|workday consultant|oracle (fusion|apps|erp)|\berp consultant\b|\bcrm consultant\b|peoplesoft"),
    ("Business / Systems Analyst", r"business analyst|systems analyst|system analyst|functional analyst|technical business analyst|business systems analyst"),
    ("IT Support / Sys & Network", r"it support|technical support engineer|help ?desk|desktop support|system administrator|systems administrator|network engineer|network administrator|service desk|application support|production support"),
    ("Tech Project / Product", r"technical project manager|it project manager|technical program manager|scrum master|technical product manager|product owner|delivery manager"),
    ("Software Developer", r"software engineer|software developer|application developer|full ?stack developer|backend developer|back end developer|java developer|python developer|c\+\+ developer|golang developer|node\.js developer|\.net developer|programmer|\bsde[ -]?[123]?\b|embedded software"),
]

**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
def classify_family(title):
    t = normalize_text(title)
    for family, pattern in FAMILY_RULES:
        if re.search(pattern, t, flags=re.I):
            return family
    return "Other / Non-IT or Unclear"

df["family"] = df["title_clean"].map(classify_family)
family_counts = df["family"].value_counts().rename_axis("family").to_frame("listings")
family_counts["share"] = (family_counts["listings"] / len(df)).round(4)
display(family_counts)
print("Rule-classified IT listings:", f"{df['family'].ne('Other / Non-IT or Unclear').sum():,}")

,listings,share
family,,
Other / Non-IT or Unclear,75882,0.8096
Software Developer,5969,0.0637
ERP / CRM Consultant,4054,0.0433
Data Engineer,1108,0.0118
DevOps / Cloud,1047,0.0112
QA / Testing,993,0.0106
Data Scientist / ML-AI,940,0.0100
IT Support / Sys & Network,868,0.0093
Frontend / Web,589,0.0063


Rule-classified IT listings: 17,847


In [ ]:
EXTRA_RULES = [
    ("Cybersecurity", r"security architect"),
    ("Data Engineer", r"data architect|data solution architect|snowflake developer"),
    ("QA / Testing", r"test automation engineer|quality engineer tester|test lead|quality engineering lead"),
    ("Frontend / Web", r"react ?js|angular|vue ?js|front ?end"),
    ("Software Developer", r"software development (engineer|lead)|\bsde\b|dot ?net|full ?stack|java lead|mainframe developer|mulesoft developer|node ?js developer|team leader\s*-?\s*software|developer\s*-?\s*l\d|software (engineer|developer|programmer)"),
    ("IT Architect", r"(solution|technology|technical|enterprise) architect"),
    ("IT Services - Ambiguous Role", r"^technical lead|application (lead|designer|tech support)"),
]

def classify_family_v2(title):
    fam = classify_family(title)
    if fam != "Other / Non-IT or Unclear":
        return fam
    t = normalize_text(title)
    for family, pat in EXTRA_RULES:
        if re.search(pat, t, flags=re.I):
            return family
    return fam

df["family"] = df["title_clean"].map(classify_family_v2)

fc = df["family"].value_counts().rename_axis("family").to_frame("listings")
fc["share"] = (fc["listings"] / len(df)).round(4)
display(fc)
print("IT listings:", df["family"].ne("Other / Non-IT or Unclear").sum())

,listings,share
family,,
Other / Non-IT or Unclear,71557,0.7634
Software Developer,7494,0.0800
ERP / CRM Consultant,4054,0.0433
IT Services - Ambiguous Role,1513,0.0161
Data Engineer,1306,0.0139
QA / Testing,1184,0.0126
DevOps / Cloud,1047,0.0112
Data Scientist / ML-AI,940,0.0100
IT Support / Sys & Network,868,0.0093


IT listings: 22172


### 4.1 Review ambiguous / missed titles

The rule-based label is a starting point, not ground truth. Inspect examples from every family and from `Other / Non-IT or Unclear`. Correct the taxonomy before using family labels in a report.

In [ ]:
for fam in df["family"].value_counts().index:
    n = int(df["family"].eq(fam).sum())
    print("\n###", fam, f"({n})")
    print(df.loc[df["family"].eq(fam), "title"].sample(n=min(8, n), random_state=42).to_string(index=False))


### Other / Non-IT or Unclear (71557)
              Risk Consulting Director - Offensive Security
                                          Account Executive
Production Manager (Fabrication/ Sheet Metal) For Jalandhar
              Service Officer-BRANCH BANKING-Branch Banking
                             Workday Security -Professional
          Sociology Teacher - Delhi Public School Faridabad
                                  Supplier Quality Engineer
                                              GSOC Operator

### Software Developer (7494)
                                               Application Developer
                                 Java Full Stack Developer - Kolkata
                                               Application Developer
                  Junior Software Developer at An IT Service Company
                    CNC VMC Operator & Programmer  - Nagar Road Pune
Sr. Python Developer - Chennai/ Bangalore - 10 Days Joiners Only !!!
                                    

In [ ]:
other = df[df["family"].eq("Other / Non-IT or Unclear")]
print("Top 60 titles in Other:")
print(other["title_norm"].value_counts().head(60))

kw = r"develop|engineer|software|java|python|\.net|sql|cloud|devops|tester|qa|programmer|sap|salesforce|network|architect|web|app"
miss = other[other["title_norm"].str.contains(kw, regex=True)]
print("\nPossible missed IT titles:", len(miss))
print(miss["title_norm"].value_counts().head(60))

Top 60 titles in Other:
title_norm
sales executive                          580
business development executive           452
business development manager             324
sales manager                            319
relationship manager                     228
accountant                               216
area sales manager                       212
customer support executive               210
account executive                        204
graphic designer                         191
business development associate           156
sales officer home loan                  152
customer care executive                  136
field sales executive                    131
project manager                          122
us it lead recruiter                     117
relationship manager - banca channel     116
customer service executive               110
marketing executive                      107
digital marketing executive              105
senior accountant                        102
mis executive       

patch 2

In [ ]:
EXTRA_RULES_2 = [
    ("Software Developer", r"\.net|dot ?net|\bmern\b|\bmean stack\b|sql developer|as400|java architect|shopify developer|senior engineer ?-? ?software|software (engineer|developer)"),
    ("Data Engineer", r"big data developer|databricks|\betl\b"),
    ("Data Scientist / ML-AI", r"large language model|\bllm\b|generative ai|gen ?ai"),
    ("QA / Testing", r"qa automation|test engineer|\bsdet\b|quality assurance"),
    ("IT Support / Sys & Network", r"technology support|system(s)? administrator|sas administrator|helpdesk"),
    ("Cybersecurity", r"offensive security"),
]

# Ye words aaye to job IT nahi hai (false positives hatane ke liye)
NEGATIVE = (r"\bcnc\b|\bvmc\b|\bplc\b|machine operator|\bautocad\b|\bmechanical\b|\bcivil\b|"
            r"\belectrical\b|\binstrumentation\b|recruiter|talent acquisition|"
            r"business development|\bsales\b|\bteacher\b")

def classify_family_v3(title):
    t = normalize_text(title)
    fam = classify_family_v2(title)
    if re.search(NEGATIVE, t):
        return "Other / Non-IT or Unclear"
    if fam != "Other / Non-IT or Unclear":
        return fam
    for family, pat in EXTRA_RULES_2:
        if re.search(pat, t, flags=re.I):
            return family
    return fam

df["family"] = df["title_clean"].map(classify_family_v3)

# Full-stack ko Software Developer mein daalo (Frontend/Web se nikaalke)
m = df["family"].eq("Frontend / Web") & df["title_norm"].str.contains(r"full ?stack")
df.loc[m, "family"] = "Software Developer"

fc = df["family"].value_counts().rename_axis("family").to_frame("listings")
fc["share"] = (fc["listings"] / len(df)).round(4)
display(fc)
print("IT listings:", df["family"].ne("Other / Non-IT or Unclear").sum())

,listings,share
family,,
Other / Non-IT or Unclear,70815,0.7555
Software Developer,7909,0.0844
ERP / CRM Consultant,4013,0.0428
IT Services - Ambiguous Role,1513,0.0161
Data Engineer,1444,0.0154
QA / Testing,1256,0.0134
Data Scientist / ML-AI,1093,0.0117
DevOps / Cloud,1047,0.0112
IT Support / Sys & Network,955,0.0102


IT listings: 22914


## 5. Optional O*NET reference loading

O*NET helps connect Indian listing titles to standardized occupational titles and alternate titles. It is **not** used as the source of Indian job listings. We use exact normalized title matches as a conservative baseline; fuzzy/embedding matches should be manually validated before being trusted.

In [ ]:
print("O*NET already loaded:", list(onet.keys()))

O*NET already loaded: ['occupation_data.csv', 'job_titles.csv', 'software_skills.csv', 'career_interest_types.csv', 'knowledge.csv', 'related_occupations.csv', 'job_zones.csv']


In [ ]:
onet = {}
ONET_FOLDER = Path(DRIVE_BASE) / "onet"
for filename in ["occupation_data.csv", "job_titles.csv", "software_skills.csv",
                 "career_interest_types.csv", "knowledge.csv",
                 "related_occupations.csv", "job_zones.csv"]:
    onet[filename] = pd.read_csv(ONET_FOLDER / filename)
print({k: v.shape for k, v in onet.items()})

{'occupation_data.csv': (1016, 3), 'job_titles.csv': (54269, 5), 'software_skills.csv': (31821, 7), 'career_interest_types.csv': (8307, 9), 'knowledge.csv': (60060, 15), 'related_occupations.csv': (18460, 6), 'job_zones.csv': (923, 5)}


In [ ]:
def norm_title(s):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9+# ]", " ", str(s).lower())).strip()

soc_lookup = {}
if "occupation_data.csv" in onet:
    occ = onet["occupation_data.csv"]
    if {"O*NET-SOC Code", "Title"}.issubset(occ.columns):
        for _, row in occ.iterrows():
            key = norm_title(row["Title"])
            if key:
                soc_lookup.setdefault(key, (row["O*NET-SOC Code"], row["Title"]))
if "job_titles.csv" in onet:
    jt = onet["job_titles.csv"]
    if {"O*NET-SOC Code", "Job Title"}.issubset(jt.columns):
        for _, row in jt.iterrows():
            key = norm_title(row["Job Title"])
            if key:
                soc_lookup.setdefault(key, (row["O*NET-SOC Code"], row["Job Title"]))

df["onet_soc_exact"] = df["title_clean"].map(lambda x: soc_lookup.get(norm_title(x), (None, None))[0])
df["onet_title_exact"] = df["title_clean"].map(lambda x: soc_lookup.get(norm_title(x), (None, None))[1])
print("Exact O*NET title matches:", f"{df['onet_soc_exact'].notna().sum():,}",
      f"({df['onet_soc_exact'].notna().mean():.1%})")
display(df.loc[df["onet_soc_exact"].notna(), ["title", "onet_title_exact", "onet_soc_exact"]].head(15))

Exact O*NET title matches: 12,852 (13.7%)


,title,onet_title_exact,onet_soc_exact
6,Maintenance Technician,Maintenance Technician,47-4051.00
9,Site Reliability Engineer,Site Reliability Engineer,15-1221.00
25,Embryologist,Embryologist,19-1029.00
68,Business Development Associate,Business Development Associate,13-1161.00
69,Business Development Associate,Business Development Associate,13-1161.00
70,Business Development Associate,Business Development Associate,13-1161.00
71,Business Development Associate,Business Development Associate,13-1161.00
72,Business Development Associate,Business Development Associate,13-1161.00
73,Business Development Associate,Business Development Associate,13-1161.00
74,Business Development Associate,Business Development Associate,13-1161.00


## 6. Skill vocabulary and normalization

We use the skills present in the Indian listings plus a small alias map for common variants. This is data-derived and editable. Skill extraction is constrained to a vocabulary to reduce random substring matches (for example, matching `R` inside ordinary words).

In [ ]:
SKILL_ALIASES = {
    "machine learning": ["ml", "machine-learning"],
    "deep learning": ["dl", "deep-learning"],
    "natural language processing": ["nlp"],
    "computer vision": ["cv"],
    "generative ai": ["gen ai", "genai"],
    "large language models": ["llm", "llms", "large language model"],
    "scikit-learn": ["sklearn", "scikit learn"],
    "power bi": ["powerbi"],
    "microsoft excel": ["ms excel", "excel"],
    "amazon web services": ["aws"],
    "google cloud platform": ["gcp"],
    "continuous integration/continuous delivery": ["ci/cd", "cicd"],
    "javascript": ["js"],
    "typescript": ["ts"],
    "postgresql": ["postgres"],
    "node.js": ["nodejs", "node js"],
    "c++": ["cpp"],
    "c#": ["c sharp", "c-sharp"],
}

**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
def canonical_skill(s):
    x = re.sub(r"\s+", " ", str(s).lower()).strip()
    for canonical, aliases in SKILL_ALIASES.items():
        if x == canonical or x in aliases:
            return canonical
    return x

# Remove generic/noisy standalone tags; multi-word technical phrases remain eligible.
# This list is editable and should be refined after reviewing false positives.
SKILL_STOPWORDS = {
    "data", "machine", "learning", "computer", "science", "computer science", "personal", "profile",
    "project", "projects", "analysis", "basic", "undergraduate", "academic", "demo",
    "visualization", "evaluation", "precision", "recall", "structures", "rest",
    "c+", "technology", "technical", "skills", "experience", "knowledge", "management",
    "communication", "team", "work", "working", "ability", "development", "business",
    "services", "solutions", "support", "system", "systems", "process", "planning",
    "reporting", "operations", "engineering", "software", "application", "customer",
    "quality", "training", "research", "tools", "using", "related", "other", "various",
    "new", "full", "time", "role", "roles", "job", "jobs", "company", "responsible",
    "requirements", "required", "preferred", "candidate", "good", "strong"
}

**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
def clean_skill_list(skills):
    cleaned = []
    for item in skills:
        skill = canonical_skill(item)
        if skill and skill not in SKILL_STOPWORDS and len(skill) > 1:
            cleaned.append(skill)
    return list(dict.fromkeys(cleaned))

df["skills_list"] = df["skills_list"].map(clean_skill_list)
df["skills_text"] = df["skills_list"].map(lambda xs: " ".join(xs))

from collections import Counter
skill_counts_initial = Counter(skill for skills in df["skills_list"] for skill in set(skills))
# Ignore one-off tags in the resume matcher to reduce typos and accidental matches.
RARE_SKILLS = {skill for skill, count in skill_counts_initial.items() if count < 2}
df["skills_list"] = df["skills_list"].map(lambda xs: [s for s in xs if s not in RARE_SKILLS])
df["skills_text"] = df["skills_list"].map(lambda xs: " ".join(xs))
skill_counts = Counter(skill for skills in df["skills_list"] for skill in set(skills))
skill_vocab_df = pd.DataFrame(skill_counts.items(), columns=["skill", "listing_count"]).sort_values("listing_count", ascending=False)
skill_vocab = set(skill_vocab_df["skill"])
print("Unique normalized skills after noise + rare-tag filtering:", len(skill_vocab))
display(skill_vocab_df.head(40))

Unique normalized skills after noise + rare-tag filtering: 19540


,skill,listing_count
82,sales,8916
417,python,5431
197,project management,5181
47,sap,4590
74,customer service,4564
328,java,3781
332,sql,3747
312,css,3627
173,business development,3603
2,communication skills,3446


In [ ]:
from collections import Counter

it_df = df[df["family"].ne("Other / Non-IT or Unclear")]
it_counts = Counter(s for sk in it_df["skills_list"] for s in set(sk))
print("IT listings:", len(it_df), "| unique skills in IT jobs:", len(it_counts))
print("skills appearing in >=10 IT listings:", sum(1 for c in it_counts.values() if c >= 10))

print("\nTop 40 skills in IT jobs:")
print(pd.DataFrame(it_counts.most_common(40), columns=["skill","it_listings"]).to_string(index=False))

print("\nTop skills per family:")
for fam in it_df["family"].value_counts().index:
    c = Counter(s for sk in it_df.loc[it_df["family"].eq(fam), "skills_list"] for s in set(sk))
    print(f"{fam}: {[s for s,_ in c.most_common(8)]}")

IT listings: 22914 | unique skills in IT jobs: 7939
skills appearing in >=10 IT listings: 1958

Top 40 skills in IT jobs:
                                     skill  it_listings
                                    python         3715
                                       sap         3041
                                       css         2827
                                      java         2738
                                       sql         2456
                                        c#         2088
                                javascript         2038
                       amazon web services         1342
                        project management         1299
                                     agile         1299
                             microservices         1197
                   application development         1183
                               spring boot         1152
                    continuous integration         1118
                                  reac

In [ ]:
MERGE = {
    "continuous integration": "continuous integration/continuous delivery",
    "net": ".net",
    "sap s hana": "sap hana",
    "react": "react.js", "reactjs": "react.js", "react js": "react.js",
    "bi": "business intelligence",
    "ai": "artificial intelligence",
}
df["skills_list"] = df["skills_list"].map(lambda xs: list(dict.fromkeys(MERGE.get(s, s) for s in xs)))
df["skills_text"] = df["skills_list"].map(lambda xs: " ".join(xs))

# Vocabulary: sirf IT jobs mein >=10 baar aane wali skills
it_df = df[df["family"].ne("Other / Non-IT or Unclear")]
it_counts = Counter(s for sk in it_df["skills_list"] for s in set(sk))
skill_vocab = {s for s, c in it_counts.items() if c >= 10}

print("Final IT skill vocabulary:", len(skill_vocab))
for s in [".net", "react.js", "continuous integration/continuous delivery", "sap hana", "python", "selenium"]:
    print(s, "->", s in skill_vocab, it_counts.get(s, 0))

Final IT skill vocabulary: 1951
.net -> True 669
react.js -> True 1210
continuous integration/continuous delivery -> True 1469
sap hana -> True 1009
python -> True 3715
selenium -> True 461


In [ ]:
import pickle
Path(DRIVE_BASE, "processed").mkdir(exist_ok=True)
df.to_pickle(Path(DRIVE_BASE) / "processed" / "jobs_checkpoint_v1.pkl")
pickle.dump(skill_vocab, open(Path(DRIVE_BASE) / "processed" / "skill_vocab_v1.pkl", "wb"))
print("saved", len(df), "rows,", len(skill_vocab), "skills")

saved 93729 rows, 1951 skills


## 7. Resume input and skill extraction

You can upload a PDF/DOCX resume or paste text manually. Extracted text is processed locally in this notebook. Review the detected skills before trusting the recommendations. A clearly labelled demo profile is included so the full notebook can run once before you replace it with your own resume.

In [ ]:
# Install parser libraries in Colab if needed. Safe to rerun.
%pip -q install pypdf python-docx scikit-learn

In [ ]:
def read_resume_file(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(str(path))
        return "\\n".join(page.extract_text() or "" for page in reader.pages)
    elif suffix == ".docx":
        from docx import Document
        doc = Document(str(path))
        parts = [p.text for p in doc.paragraphs]
        for table in doc.tables:
            for row in table.rows:
                parts.append(" | ".join(cell.text for cell in row.cells))
        return "\\n".join(parts)
    elif suffix in {".txt", ".md"}:
        return path.read_text(encoding="utf-8", errors="ignore")
    else:
        raise ValueError("Supported formats: PDF, DOCX, TXT, MD")

**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
def extract_skills(text, vocabulary=None):
    text_norm = normalize_text(text)
    vocab = vocabulary or skill_vocab
    found = set()
    # Match longer/multi-word phrases first. Boundaries reduce accidental substring matches.
    for skill in sorted(vocab, key=len, reverse=True):
        if not skill or len(skill) < 2:
            continue
        pattern = r"(?<![a-z0-9])" + re.escape(skill) + r"(?![a-z0-9])"
        if re.search(pattern, text_norm):
            found.add(skill)
    return sorted(found)

# Option A: paste resume text into the triple quotes.
RESUME_TEXT = """
DEMO PROFILE (replace with your own resume text before personal use):
Computer Science undergraduate with Python, C++, SQL, Git and data structures.
Academic projects: exploratory data analysis with pandas and NumPy; machine learning
classification using scikit-learn; model evaluation with precision, recall and F1-score.
Familiar with statistics, data visualization, REST APIs and basic Linux.
"""

# Option B: upload resume in Colab and set RESUME_PATH to its exact path.
RESUME_PATH = None  # Example: "/content/my_resume.pdf"

**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
if RESUME_PATH and Path(RESUME_PATH).exists():
    RESUME_TEXT = read_resume_file(RESUME_PATH)

resume_text_clean = clean_text(RESUME_TEXT)
resume_skills = extract_skills(resume_text_clean)
print("Resume text characters:", len(resume_text_clean))
print("Detected skills:", len(resume_skills))
display(pd.DataFrame({"detected_resume_skills": resume_skills}))

Resume text characters: 395
Detected skills: 15


,detected_resume_skills
0,apis
1,c++
2,data analysis
3,data structures
4,data visualization
5,git
6,linux
7,machine learning
8,numpy
9,pandas


## 8. Target preferences
Change these values before generating recommendations. Leave unknown preferences as `None`.

In [ ]:
TARGET_ROLE = None       # Examples: "Data Scientist / ML-AI", "Software Developer", "Cybersecurity"; None = all IT families
PREFERRED_CITY = None    # Example: "Bengaluru"; None = any city
USER_EXPERIENCE_YEARS = None  # Example: 0 for fresher; None = unknown
TOP_N = 15
MIN_TEXT_LENGTH = 15

## 9. Build the recommendation index

Ranking uses a transparent weighted score:
- **Text relevance**: TF-IDF cosine similarity between the resume and job title/skills/description.
- **Skill coverage**: fraction of a listing's extracted skills found in the resume.
- **Experience fit**: gentle preference for jobs whose stated experience range is compatible with the candidate's years; missing experience is neutral.
- **Role/city filters**: by default, only listings classified into IT families are eligible; setting a role or city narrows that pool.

Weights are configurable. Scores are relative ranking signals, **not** probabilities or hiring chances.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Keep records with usable titles. Preserve the full cleaned dataframe separately.
jobs = df[df["title_norm"].ne("")].copy().reset_index(drop=True)

def build_job_document(row):
    title = row.get("title_clean", "")
    family = row.get("family", "")
    skills = row.get("skills_text", "")
    desc = row.get("description_clean", "")
    # Repeat title and tags to give them greater influence than long boilerplate descriptions.
    return f"{title} {title} {family} {skills} {skills} {desc[:5000]}"

jobs["job_document"] = jobs.apply(build_job_document, axis=1)
vectorizer = TfidfVectorizer(
    lowercase=True, ngram_range=(1, 2), min_df=2, max_features=60000,
    sublinear_tf=True, strip_accents="unicode", token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9+#.\-]*\b"
)
job_matrix = vectorizer.fit_transform(jobs["job_document"].fillna(""))
print("Indexed jobs:", len(jobs))
print("TF-IDF matrix:", job_matrix.shape, "| non-zero values:", job_matrix.nnz)

Indexed jobs: 93729
TF-IDF matrix: (93729, 60000) | non-zero values: 10536598


In [ ]:
def recommend_jobs(resume_text, resume_skills=None, target_role=None, preferred_city=None,
                   experience_years=None, top_n=15, text_weight=0.55, skill_weight=0.35,
                   experience_weight=0.10):
    resume_text = clean_text(resume_text)
    resume_skills = set(resume_skills if resume_skills is not None else extract_skills(resume_text))
    if len(resume_text.strip()) < MIN_TEXT_LENGTH and not resume_skills:
        raise ValueError("Add resume text or detected skills before requesting recommendations.")

    candidate_mask = jobs["family"].ne("Other / Non-IT or Unclear")
    if target_role:
        candidate_mask &= jobs["family"].eq(target_role)
    if preferred_city:
        candidate_mask &= jobs["city"].str.lower().eq(preferred_city.lower())
    candidate_idx = np.flatnonzero(candidate_mask.to_numpy())
    if len(candidate_idx) == 0:
        return pd.DataFrame(), {"message": "No listings match the selected role/city filters."}

    resume_vec = vectorizer.transform([resume_text])
    text_scores = cosine_similarity(resume_vec, job_matrix[candidate_idx]).ravel()

    skill_scores = []
    missing_skills_list = []
    matched_skills_list = []
    for idx in candidate_idx:
        required = set(jobs.iloc[idx]["skills_list"])
        matched = sorted(required & resume_skills)
        missing = sorted(required - resume_skills)
        # If no job skills are provided, don't penalize it as a zero-coverage job.
        coverage = (len(matched) / len(required)) if required else 0.0
        skill_scores.append(coverage)
        matched_skills_list.append(matched)
        missing_skills_list.append(missing)
    skill_scores = np.asarray(skill_scores, dtype=float)

    exp_scores = np.full(len(candidate_idx), 0.5, dtype=float)  # neutral when unknown
    if experience_years is not None:
        exp_min = jobs.iloc[candidate_idx]["experience_min"].to_numpy(dtype=float)
        exp_max = jobs.iloc[candidate_idx]["experience_max"].to_numpy(dtype=float)
        for j, (lo, hi) in enumerate(zip(exp_min, exp_max)):
            if np.isnan(lo) and np.isnan(hi):
                exp_scores[j] = 0.5
            elif (not np.isnan(lo)) and experience_years < lo:
                exp_scores[j] = max(0.0, 1 - (lo - experience_years) / max(lo, 1))
            elif (not np.isnan(hi)) and experience_years > hi:
                exp_scores[j] = max(0.0, 1 - (experience_years - hi) / max(hi, 1))
            else:
                exp_scores[j] = 1.0

    # Renormalize the configured weights if they are changed.
    total_w = text_weight + skill_weight + experience_weight
    score = (text_weight * text_scores + skill_weight * skill_scores + experience_weight * exp_scores) / total_w

    out = jobs.iloc[candidate_idx].copy()
    out["text_similarity"] = text_scores
    out["skill_coverage"] = skill_scores
    out["experience_fit"] = exp_scores
    out["recommendation_score"] = score
    out["matched_resume_skills"] = matched_skills_list
    out["missing_skills"] = missing_skills_list
    out["required_skill_count"] = out["skills_list"].map(len)
    out["matched_skill_count"] = out["matched_resume_skills"].map(len)
    out["missing_skill_count"] = out["missing_skills"].map(len)
    out = out.sort_values(["recommendation_score", "text_similarity"], ascending=False).head(top_n)

    cols = [c for c in [
        "title", "companyName", "location", "city", "family", "experience", "salary",
        "text_similarity", "skill_coverage", "experience_fit", "recommendation_score",
        "matched_resume_skills", "missing_skills", "jobDescription", "jobId"
    ] if c in out.columns]
    return out[cols].reset_index(drop=True), {
        "candidate_jobs_considered": int(len(candidate_idx)),
        "resume_skill_count": len(resume_skills),
        "resume_skills": sorted(resume_skills),
        "weights": {"text": text_weight, "skills": skill_weight, "experience": experience_weight},
    }

In [ ]:
import numpy as np, pandas as pd
from collections import Counter

# ---------- 1) Family-level market profiles ----------
IT_FAMS = [f for f in df["family"].unique() if f != "Other / Non-IT or Unclear"]
it_df = df[df["family"].isin(IT_FAMS)]
N_IT = len(it_df)
it_doc = Counter(s for sk in it_df["skills_list"] for s in set(sk))

fam_profile = {}
for fam in IT_FAMS:
    sub = it_df[it_df["family"].eq(fam)]
    n = len(sub)
    c = Counter(s for sk in sub["skills_list"] for s in set(sk))
    rows = []
    for s, k in c.items():
        if k < 5:
            continue
        p_f = k / n
        spec = 1 + np.log(N_IT / it_doc[s])
        rows.append((s, p_f, p_f * spec))
    prof = (pd.DataFrame(rows, columns=["skill", "share_in_family", "weight"])
              .sort_values("weight", ascending=False).head(25).reset_index(drop=True))
    fam_profile[fam] = prof

def family_match(resume_skills):
    rs = set(resume_skills)
    out = []
    for fam, prof in fam_profile.items():
        have = prof["skill"].isin(rs)
        out.append((fam, prof.loc[have, "weight"].sum() / prof["weight"].sum(),
                    int(have.sum()), len(prof)))
    res = pd.DataFrame(out, columns=["family", "match", "skills_have", "profile_size"])
    res["match"] = res["match"].round(3)
    return res.sort_values("match", ascending=False).reset_index(drop=True)

def family_gap(resume_skills, fam, top_n=10):
    prof = fam_profile[fam]
    miss = prof[~prof["skill"].isin(set(resume_skills))].head(top_n).copy()
    miss["pct_of_family_listings"] = (miss["share_in_family"] * 100).round(1)
    return miss[["skill", "pct_of_family_listings"]].reset_index(drop=True)

# ---------- 2) Fixed job ranking ----------
def recommend_jobs_v2(resume_text, resume_skills, experience_years=None, top_n=10, k=3, **kw):
    r, info = recommend_jobs(resume_text, resume_skills, experience_years=experience_years,
                             top_n=10**6, **kw)
    n_match = r["matched_resume_skills"].map(len)
    n_req = n_match + r["missing_skills"].map(len)
    r["skill_smoothed"] = n_match / (n_req + k)          # few-tag jobs no longer get 100%
    text_rel = r["text_similarity"] / max(r["text_similarity"].max(), 1e-9)
    base = 0.6 * text_rel + 0.4 * r["skill_smoothed"]
    if experience_years is not None:                       # experience acts as a real penalty
        base = base * (0.3 + 0.7 * r["experience_fit"])
    r["score_v2"] = base
    return r.sort_values("score_v2", ascending=False).head(top_n).reset_index(drop=True)

In [ ]:
print(family_match(resume_skills).to_string())

top = recommend_jobs_v2(resume_text_clean, resume_skills, experience_years=0, top_n=10)
print(top[["title","family","experience","text_similarity","skill_smoothed",
           "experience_fit","score_v2"]].to_string())

print(family_gap(resume_skills, "Data Scientist / ML-AI").to_string())

                          family  match  skills_have  profile_size
0         Data Scientist / ML-AI  0.296            5            25
1              Data / BI Analyst  0.269            4            25
2                  Data Engineer  0.138            2            25
3             Software Developer  0.116            3            25
4   IT Services - Ambiguous Role  0.093            3            25
5     IT Support / Sys & Network  0.080            2            25
6                 DevOps / Cloud  0.078            2            25
7     Business / Systems Analyst  0.072            2            25
8                 Database Admin  0.035            1            25
9                   QA / Testing  0.029            1            25
10                Frontend / Web  0.028            1            25
11              Mobile Developer  0.028            1            25
12                  IT Architect  0.027            1            25
13          ERP / CRM Consultant  0.000            0          

## 10. Generate and inspect recommendations

In [ ]:
recommendations, recommendation_info = recommend_jobs(
    resume_text=resume_text_clean,
    resume_skills=resume_skills,
    target_role=TARGET_ROLE,
    preferred_city=PREFERRED_CITY,
    experience_years=USER_EXPERIENCE_YEARS,
    top_n=TOP_N,
)
print(recommendation_info)
if not recommendations.empty:
    display(recommendations.drop(columns=["jobDescription"], errors="ignore"))
else:
    print(recommendation_info.get("message", "No recommendations."))

{'candidate_jobs_considered': 22914, 'resume_skill_count': 15, 'resume_skills': ['apis', 'c++', 'data analysis', 'data structures', 'data visualization', 'git', 'linux', 'machine learning', 'numpy', 'pandas', 'python', 'rest apis', 'scikit-learn', 'sql', 'statistics'], 'weights': {'text': 0.55, 'skills': 0.35, 'experience': 0.1}}


,title,companyName,location,city,family,experience,salary,text_similarity,skill_coverage,experience_fit,recommendation_score,matched_resume_skills,missing_skills,jobId
0,Senior Data Scientist,Piramal Finance,"Mumbai, Bengaluru",Mumbai,Data Scientist / ML-AI,3-5 Yrs,10-20 Lacs PA,0.254071,1.000000,0.5,0.539739,"[data analysis, data visualization, machine learning, python, statistics]",[],290925012742
1,Machine Learning Engineer,Capgemini,"Hybrid - Bengaluru, Delhi / NCR, Mumbai (All Areas)",Bengaluru,Data Scientist / ML-AI,6-11 Yrs,Not disclosed,0.115666,1.000000,0.5,0.463616,"[machine learning, python, sql]",[],300925013246
2,Data Analyst-Visualization,Shyam Dairy (Shyam Group),Prayagraj,Prayagraj,Data / BI Analyst,3-5 Yrs,Not disclosed,0.105162,1.000000,0.5,0.457839,"[data analysis, data visualization]",[],300925013071
3,Data Scientist - 3+ Years - Bangalore/Gurgaon,Crescendo Global Leadership Hiring India,"Hybrid - Bengaluru, Delhi / NCR",Bengaluru,Data Scientist / ML-AI,3-7 Yrs,Not disclosed,0.095237,1.000000,0.5,0.452380,"[machine learning, python, sql]",[],011025021138
4,Python Application Developer,Launch IT India,Bengaluru,Bengaluru,Software Developer,5-10 Yrs,Not disclosed,0.072193,1.000000,0.5,0.439706,"[git, python, sql]",[],290925005890
5,C++ Developer,Venpa Staffing,Bengaluru,Bengaluru,Software Developer,4-6 Yrs,10-16 Lacs PA,0.070288,1.000000,0.5,0.438658,"[c++, linux, python]",[],300925027776
6,GenAI Developer,Dharani Info Technologies,Guntur,Guntur,Data Scientist / ML-AI,1-4 Yrs,Not disclosed,0.067483,1.000000,0.5,0.437116,"[machine learning, python]",[],011025503168
7,Data Scientist,NKT Operations,Chennai,Chennai,Data Scientist / ML-AI,5-10 Yrs,Not disclosed,0.208597,0.750000,0.5,0.427228,"[machine learning, numpy, pandas, python, scikit-learn, statistics]","[azure, data science]",300925028671
8,ETL & Database Testers Chennai / Pune,White Horse Manpower Consultancy,"Pune, Chennai",Pune,Data Engineer,2-7 Yrs,4-7 Lacs PA,0.021446,1.000000,0.5,0.411795,[sql],[],300925024835
9,Data Scientist Jr/Sr ...Singapore/ India,DE Recruitment Mumbai,"Hybrid - Hyderabad, Pune, Bengaluru",Hyderabad,Data Scientist / ML-AI,0-5 Yrs,11-19 Lacs PA (Including Variable: 30.0%),0.112423,0.833333,0.5,0.403499,"[data analysis, machine learning, python, sql, statistics]",[etl],021025007844


In [ ]:
cols = [c for c in ["title","companyName","city","family","experience",
                    "recommendation_score","matched_resume_skills","missing_skills"]
        if c in recommendations.columns]
print(recommendations[cols].head(10).to_string())

                                           title                               companyName       city                  family experience  recommendation_score                                                      matched_resume_skills         missing_skills
0                          Senior Data Scientist                           Piramal Finance     Mumbai  Data Scientist / ML-AI    3-5 Yrs              0.539739  [data analysis, data visualization, machine learning, python, statistics]                     []
1                      Machine Learning Engineer                                 Capgemini  Bengaluru  Data Scientist / ML-AI   6-11 Yrs              0.463616                                            [machine learning, python, sql]                     []
2                     Data Analyst-Visualization                 Shyam Dairy (Shyam Group)  Prayagraj       Data / BI Analyst    3-5 Yrs              0.457839                                        [data analysis, data visualizat

In [ ]:
rec0, info0 = recommend_jobs(resume_text=resume_text_clean, resume_skills=resume_skills,
                             experience_years=0, top_n=10)
print(rec0[["title","family","experience","text_similarity","skill_coverage",
            "experience_fit","recommendation_score"]].to_string())

                                           title                  family experience  text_similarity  skill_coverage  experience_fit  recommendation_score
0                          Senior Data Scientist  Data Scientist / ML-AI    3-5 Yrs         0.254071        1.000000             0.0              0.489739
1       Data Scientist Jr/Sr ...Singapore/ India  Data Scientist / ML-AI    0-5 Yrs         0.112423        0.833333             1.0              0.453499
2                      Machine Learning Engineer  Data Scientist / ML-AI   6-11 Yrs         0.115666        1.000000             0.0              0.413616
3                     Data Analyst-Visualization       Data / BI Analyst    3-5 Yrs         0.105162        1.000000             0.0              0.407839
4  Data Scientist - 3+ Years - Bangalore/Gurgaon  Data Scientist / ML-AI    3-7 Yrs         0.095237        1.000000             0.0              0.402380
5                   Python Application Developer      Software Develop

## 11. Detailed skill-gap analysis for a selected job

Choose a recommendation row. The comparison is between the skills extracted from the resume and the skills explicitly tagged in that job listing. If the job has no tags, its skill-gap report will be sparse; the title/description still contributes to text relevance.

In [ ]:
if not recommendations.empty:
    SELECTED_JOB_ROW = 0
    selected = recommendations.iloc[SELECTED_JOB_ROW]
    print("Selected role:", selected["title"], "| Company:", selected["companyName"])
    print("Recommendation score:", round(float(selected["recommendation_score"]), 3))
    print("Matched skills:", selected["matched_resume_skills"])
    print("Missing / not detected skills:", selected["missing_skills"])
else:
    print("Run the recommendation cell after providing resume text.")

Selected role: Senior Data Scientist | Company: Piramal Finance
Recommendation score: 0.54
Matched skills: ['data analysis', 'data visualization', 'machine learning', 'python', 'statistics']
Missing / not detected skills: []


## 12. Personalized learning roadmap

The roadmap uses two signals:
1. **Market demand**: how often a skill appears among listings in the target role/family.
2. **Prerequisite ordering**: a small transparent graph for common technical skills.

The prerequisite graph is a starter knowledge base, not a universal curriculum. Unknown skills are placed after known prerequisites and prioritized by target-role demand. Edit the graph for your chosen career path.

In [ ]:
PREREQUISITES = {
    "python": [],
    "sql": [],
    "statistics": [],
    "linear algebra": ["statistics"],
    "numpy": ["python"],
    "pandas": ["python", "numpy"],
    "scikit-learn": ["python", "pandas", "statistics"],
    "machine learning": ["python", "statistics", "scikit-learn"],
    "deep learning": ["python", "linear algebra", "machine learning"],
    "pytorch": ["python", "deep learning"],
    "tensorflow": ["python", "deep learning"],
    "natural language processing": ["python", "machine learning"],
    "generative ai": ["python", "machine learning", "natural language processing"],
    "large language models": ["python", "natural language processing", "deep learning"],
    "power bi": ["sql", "microsoft excel"],
    "tableau": ["sql"],
    "git": [],
    "linux": [],
    "docker": ["linux"],
    "kubernetes": ["docker", "linux"],
    "aws": ["linux"],
    "amazon web services": ["linux"],
    "azure": ["linux"],
    "google cloud platform": ["linux"],
    "spark": ["python", "sql"],
    "apache spark": ["python", "sql"],
    "airflow": ["python", "sql"],
    "data engineering": ["python", "sql"],
    "mongodb": [],
    "postgresql": ["sql"],
    "javascript": [],
    "html": [],
    "css": ["html"],
    "react": ["javascript", "html", "css"],
    "node.js": ["javascript"],
    "java": [],
    "c++": [],
    "networking": [],
    "cybersecurity": ["networking", "linux"],
}

**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
def build_learning_roadmap(missing_skills, target_family=None, max_items=15):
    missing = {canonical_skill(s) for s in missing_skills}
    # Add known prerequisites if the missing skill depends on them and they were not detected in the resume.
    expanded = set(missing)
    changed = True
    while changed:
        changed = False
        for skill in list(expanded):
            for prereq in PREREQUISITES.get(skill, []):
                prereq = canonical_skill(prereq)
                if prereq not in resume_skills and prereq not in expanded:
                    expanded.add(prereq)
                    changed = True

    # Demand estimates among listings in the selected family (or all IT families).
    if target_family:
        demand_df = df[df["family"].eq(target_family)]
    else:
        demand_df = df[df["family"].ne("Other / Non-IT or Unclear")]
    demand = Counter(skill for skills in demand_df["skills_list"] for skill in set(skills))

    # Simple topological ordering; any cycles/unrecognized skills fall back to demand.
    ordered, remaining = [], set(expanded)
    while remaining:
        ready = [s for s in remaining
                 if all(canonical_skill(p) not in remaining for p in PREREQUISITES.get(s, []))]
        if not ready:
            ready = list(remaining)
        ready.sort(key=lambda s: (demand.get(s, 0), s), reverse=True)
        for s in ready:
            ordered.append(s)
            remaining.remove(s)

    roadmap = []
    for rank, skill in enumerate(ordered[:max_items], start=1):
        deps = [canonical_skill(p) for p in PREREQUISITES.get(skill, [])]
        roadmap.append({
            "priority": rank,
            "skill_to_learn": skill,
            "market_listing_mentions": int(demand.get(skill, 0)),
            "known_prerequisites": ", ".join(deps) if deps else "No prerequisite recorded",
            "reason": "Required by a recommended role / added as a prerequisite" if skill in missing else "Prerequisite added by roadmap graph",
        })
    return pd.DataFrame(roadmap)

**Checkpoint:** Run this cell, inspect its output/variables, and only then continue to the next code cell. If the result is unexpected, stop and share it before proceeding.

In [ ]:
if not recommendations.empty:
    selected = recommendations.iloc[SELECTED_JOB_ROW]
    roadmap = build_learning_roadmap(
        missing_skills=selected["missing_skills"],
        target_family=selected["family"] if selected["family"] != "Other / Non-IT or Unclear" else None,
    )
    display(roadmap)
else:
    roadmap = pd.DataFrame()
    print("No roadmap until at least one recommendation is available.")

""


In [ ]:
print(roadmap.head(10).to_string())

Empty DataFrame
Columns: []
Index: []


In [ ]:
Path(DRIVE_BASE, "processed").mkdir(exist_ok=True)
df.to_pickle(Path(DRIVE_BASE) / "processed" / "jobs_checkpoint_v2.pkl")
print("saved", len(df))

saved 93729


In [ ]:
gap = family_gap(resume_skills, "Data Scientist / ML-AI", top_n=12)
roadmap = build_learning_roadmap(gap["skill"].tolist(), target_family="Data Scientist / ML-AI")
print(roadmap.head(15).to_string())

    priority               skill_to_learn  market_listing_mentions                                    known_prerequisites                                                    reason
0          1      artificial intelligence                      334                               No prerequisite recorded  Required by a recommended role / added as a prerequisite
1          2  natural language processing                      277                               python, machine learning  Required by a recommended role / added as a prerequisite
2          3                 data science                      209                               No prerequisite recorded  Required by a recommended role / added as a prerequisite
3          4                   algorithms                      122                               No prerequisite recorded  Required by a recommended role / added as a prerequisite
4          5             image processing                      110                               No 

In [ ]:
import numpy as np, pandas as pd
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize
from sklearn.metrics import f1_score

AMBIG = "IT Services - Ambiguous Role"
ev = df[df["family"].ne("Other / Non-IT or Unclear") & df["family"].ne(AMBIG)].copy()
ev = ev[ev["skills_list"].map(len) >= 3].reset_index(drop=True)

# Same company ki jobs train aur test dono mein na jaayein (leakage se bachne ke liye)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(ev, groups=ev["companyName"].fillna("unk")))
tr, te = ev.iloc[tr_idx], ev.iloc[te_idx]
fams = sorted(ev["family"].unique())
y = te["family"].to_numpy()
print("train:", len(tr), "| test:", len(te), "| families:", len(fams))

def evaluate(S, name):
    S = np.asarray(S)
    order = np.argsort(-S, axis=1)
    fa = np.array(fams)
    top1, top3 = fa[order[:, 0]], fa[order[:, :3]]
    return {"model": name,
            "top1_acc": round((top1 == y).mean(), 3),
            "top3_acc": round(np.mean([y[i] in top3[i] for i in range(len(y))]), 3),
            "macro_F1": round(f1_score(y, top1, average="macro"), 3)}

results = []

# 0) Baseline: hamesha sabse common family
cnt = tr["family"].value_counts()
S0 = np.tile([cnt.get(f, 0) for f in fams], (len(te), 1)).astype(float)
results.append(evaluate(S0, "Baseline: most common family"))

# A) Weighted skill-profile match (jo humne banaya)
def build_profiles(train, top_k=25, min_count=5):
    N = len(train)
    doc = Counter(s for sk in train["skills_list"] for s in set(sk))
    prof = {}
    for fam in fams:
        sub = train[train["family"].eq(fam)]
        n = len(sub)
        c = Counter(s for sk in sub["skills_list"] for s in set(sk))
        w = {s: (k / n) * (1 + np.log(N / doc[s])) for s, k in c.items() if k >= min_count}
        top = dict(sorted(w.items(), key=lambda x: -x[1])[:top_k])
        prof[fam] = (top, sum(top.values()))
    return prof
prof = build_profiles(tr)
SA = np.array([[sum(w for s, w in prof[f][0].items() if s in set(sk)) / prof[f][1] for f in fams]
               for sk in te["skills_list"]])
results.append(evaluate(SA, "A) Skill-profile match (ours)"))

# B) TF-IDF centroid cosine
tf = TfidfVectorizer(analyzer=lambda x: x, min_df=3, sublinear_tf=True)
Ttr, Tte = tf.fit_transform(tr["skills_list"]), tf.transform(te["skills_list"])
cent = normalize(np.vstack([np.asarray(Ttr[(tr["family"] == f).to_numpy()].mean(axis=0)) for f in fams]))
results.append(evaluate(Tte @ cent.T, "B) TF-IDF centroid cosine"))

# C) Logistic Regression
cv = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3)
Xtr, Xte = cv.fit_transform(tr["skills_list"]), cv.transform(te["skills_list"])
lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, tr["family"])
assert list(lr.classes_) == fams
results.append(evaluate(lr.predict_proba(Xte), "C) Logistic Regression (balanced)"))

print(pd.DataFrame(results).to_string(index=False))

train: 17799 | test: 3457 | families: 15
                            model  top1_acc  top3_acc  macro_F1
     Baseline: most common family     0.336     0.612     0.034
    A) Skill-profile match (ours)     0.635     0.823     0.579
        B) TF-IDF centroid cosine     0.685     0.914     0.625
C) Logistic Regression (balanced)     0.758     0.931     0.687


In [ ]:
from sklearn.metrics import classification_report

pred = lr.predict(Xte)
print(classification_report(y, pred, digits=3))

cm = pd.crosstab(pd.Series(y, name="true"), pd.Series(pred, name="pred"))
pairs = [(t, p, int(cm.loc[t, p])) for t in cm.index for p in cm.columns if t != p]
top_conf = pd.DataFrame(sorted(pairs, key=lambda x: -x[2])[:12],
                        columns=["true_family", "predicted_as", "count"])
print(top_conf.to_string(index=False))

pd.DataFrame(results).to_csv(Path(DRIVE_BASE) / "processed" / "family_eval_results_v1.csv", index=False)

                            precision    recall  f1-score   support

Business / Systems Analyst      0.505     0.658     0.571        79
             Cybersecurity      0.785     0.849     0.816        73
         Data / BI Analyst      0.542     0.776     0.638        58
             Data Engineer      0.816     0.813     0.814       283
    Data Scientist / ML-AI      0.745     0.791     0.767       177
            Database Admin      0.549     0.867     0.672        45
            DevOps / Cloud      0.547     0.766     0.638       167
      ERP / CRM Consultant      0.919     0.849     0.883       669
            Frontend / Web      0.486     0.726     0.582       164
              IT Architect      0.217     0.357     0.270        56
IT Support / Sys & Network      0.698     0.685     0.691       165
          Mobile Developer      0.684     0.830     0.750        47
              QA / Testing      0.813     0.868     0.839       250
        Software Developer      0.902     0.694

In [ ]:
from sklearn.model_selection import GroupKFold
from scipy.sparse import hstack

def run_cv(data, label_col="family", use_desc=False, class_weight="balanced", n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    groups = data["companyName"].fillna("unk")
    t1, t3, f1s = [], [], []
    for tr_i, va_i in gkf.split(data, groups=groups):
        a, b = data.iloc[tr_i], data.iloc[va_i]
        cvz = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3)
        Xa, Xb = cvz.fit_transform(a["skills_list"]), cvz.transform(b["skills_list"])
        if use_desc:
            tfd = TfidfVectorizer(max_features=20000, min_df=5, sublinear_tf=True,
                                  ngram_range=(1, 2), stop_words="english")
            Da, Db = tfd.fit_transform(a["description_clean"]), tfd.transform(b["description_clean"])
            Xa, Xb = hstack([Xa, Da]).tocsr(), hstack([Xb, Db]).tocsr()
        m = LogisticRegression(max_iter=1000, class_weight=class_weight).fit(Xa, a[label_col])
        P, cls = m.predict_proba(Xb), m.classes_
        order = np.argsort(-P, axis=1)
        yb = b[label_col].to_numpy()
        top1, top3 = cls[order[:, 0]], cls[order[:, :3]]
        t1.append((top1 == yb).mean())
        t3.append(np.mean([yb[i] in top3[i] for i in range(len(yb))]))
        f1s.append(f1_score(yb, top1, average="macro"))
    fmt = lambda v: f"{np.mean(v):.3f} ± {np.std(v):.3f}"
    return {"top1_acc": fmt(t1), "top3_acc": fmt(t3), "macro_F1": fmt(f1s)}

# Variant 4: Frontend ko Software Developer mein merge, IT Architect hata do
tr_m = tr[tr["family"].ne("IT Architect")].copy()
tr_m["family_merged"] = tr_m["family"].replace({"Frontend / Web": "Software / Web Developer",
                                                "Software Developer": "Software / Web Developer"})

cv_results = [
    {"variant": "1) Skills only (current)",           **run_cv(tr)},
    {"variant": "2) Skills + description text",       **run_cv(tr, use_desc=True)},
    {"variant": "3) Skills only, no class weights",   **run_cv(tr, class_weight=None)},
    {"variant": "4) Merged families (SW+Frontend, no Architect)", **run_cv(tr_m, label_col="family_merged")},
]
cv_df = pd.DataFrame(cv_results)
print(cv_df.to_string(index=False))
cv_df.to_csv(Path(DRIVE_BASE) / "processed" / "family_cv_results_v1.csv", index=False)

                                       variant      top1_acc      top3_acc      macro_F1
                      1) Skills only (current) 0.699 ± 0.093 0.912 ± 0.031 0.647 ± 0.089
                  2) Skills + description text 0.755 ± 0.080 0.941 ± 0.018 0.699 ± 0.086
              3) Skills only, no class weights 0.753 ± 0.070 0.934 ± 0.011 0.677 ± 0.077
4) Merged families (SW+Frontend, no Architect) 0.762 ± 0.085 0.937 ± 0.014 0.686 ± 0.073


In [ ]:
def run_cv_folds(data, label_col="family", use_desc=False, class_weight=None, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    groups = data["companyName"].fillna("unk")
    t1, t3, f1s = [], [], []
    for tr_i, va_i in gkf.split(data, groups=groups):
        a, b = data.iloc[tr_i], data.iloc[va_i]
        cvz = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3)
        Xa, Xb = cvz.fit_transform(a["skills_list"]), cvz.transform(b["skills_list"])
        if use_desc:
            tfd = TfidfVectorizer(max_features=20000, min_df=5, sublinear_tf=True,
                                  ngram_range=(1, 2), stop_words="english")
            Da, Db = tfd.fit_transform(a["description_clean"]), tfd.transform(b["description_clean"])
            Xa, Xb = hstack([Xa, Da]).tocsr(), hstack([Xb, Db]).tocsr()
        m = LogisticRegression(max_iter=1000, class_weight=class_weight).fit(Xa, a[label_col])
        P, cls = m.predict_proba(Xb), m.classes_
        order = np.argsort(-P, axis=1)
        yb = b[label_col].to_numpy()
        top1, top3 = cls[order[:, 0]], cls[order[:, :3]]
        t1.append((top1 == yb).mean())
        t3.append(np.mean([yb[i] in top3[i] for i in range(len(yb))]))
        f1s.append(f1_score(yb, top1, average="macro"))
    return {"top1": np.array(t1), "top3": np.array(t3), "f1": np.array(f1s)}

res = {
    "1) skills, balanced weights":   run_cv_folds(tr, class_weight="balanced"),
    "3) skills, no weights":         run_cv_folds(tr, class_weight=None),
    "5) skills+desc, no weights":    run_cv_folds(tr, use_desc=True, class_weight=None),
}
base = res["1) skills, balanced weights"]
rows = []
for name, v in res.items():
    d1, d3, dF = v["top1"] - base["top1"], v["top3"] - base["top3"], v["f1"] - base["f1"]
    rows.append({"variant": name,
                 "top1": f"{v['top1'].mean():.3f}", "top3": f"{v['top3'].mean():.3f}",
                 "macro_F1": f"{v['f1'].mean():.3f}",
                 "d_top1_vs_1": f"{d1.mean():+.3f} (better in {(d1 > 0).sum()}/{len(d1)} folds)",
                 "d_F1_vs_1": f"{dF.mean():+.3f} (better in {(dF > 0).sum()}/{len(dF)} folds)"})
out = pd.DataFrame(rows)
print(out.to_string(index=False))
print("\nPer-fold top1:")
for name, v in res.items():
    print(f"  {name}: {np.round(v['top1'], 3).tolist()}")
out.to_csv(Path(DRIVE_BASE) / "processed" / "family_cv_results_v2.csv", index=False)

                    variant  top1  top3 macro_F1                  d_top1_vs_1                    d_F1_vs_1
1) skills, balanced weights 0.699 0.912    0.647 +0.000 (better in 0/5 folds) +0.000 (better in 0/5 folds)
      3) skills, no weights 0.753 0.934    0.677 +0.053 (better in 5/5 folds) +0.030 (better in 5/5 folds)
 5) skills+desc, no weights 0.805 0.955    0.727 +0.105 (better in 5/5 folds) +0.080 (better in 5/5 folds)

Per-fold top1:
  1) skills, balanced weights: [0.516, 0.771, 0.752, 0.719, 0.738]
  3) skills, no weights: [0.614, 0.789, 0.8, 0.766, 0.794]
  5) skills+desc, no weights: [0.71, 0.836, 0.837, 0.81, 0.83]


In [ ]:
GENERIC_SKILLS = {"artificial intelligence", "data science", "algorithms", "data", "consulting",
    "project management", "troubleshooting", "automation", "architecture", "software development",
    "application development", "development methodologies", "technical leadership",
    "design principles", "application design", "web services", "debugging", "analytical"}

# ---- Final models (sirf TRAIN split pe fit) ----
# Model S: sirf skills -> resume ke liye
cv_s = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3)
model_s = LogisticRegression(max_iter=1000).fit(cv_s.fit_transform(tr["skills_list"]), tr["family"])

# Model D: skills + description -> job postings ke liye
cv_d = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3)
tf_d = TfidfVectorizer(max_features=20000, min_df=5, sublinear_tf=True,
                       ngram_range=(1, 2), stop_words="english")
Xd_tr = hstack([cv_d.fit_transform(tr["skills_list"]), tf_d.fit_transform(tr["description_clean"])]).tocsr()
model_d = LogisticRegression(max_iter=1000).fit(Xd_tr, tr["family"])

def X_d(frame):
    return hstack([cv_d.transform(frame["skills_list"]), tf_d.transform(frame["description_clean"])]).tocsr()

def score(model, X, y_true):
    P, cls = model.predict_proba(X), model.classes_
    order = np.argsort(-P, axis=1)
    top1, top3 = cls[order[:, 0]], cls[order[:, :3]]
    return {"top1_acc": round((top1 == y_true).mean(), 3),
            "top3_acc": round(np.mean([y_true[i] in top3[i] for i in range(len(y_true))]), 3),
            "macro_F1": round(f1_score(y_true, top1, average="macro"), 3)}

# ---- Held-out test split ----
print(pd.DataFrame([
    {"model": "S: skills only",   **score(model_s, cv_s.transform(te["skills_list"]), y)},
    {"model": "D: skills + desc", **score(model_d, X_d(te), y)},
]).to_string(index=False))

# ---- Resume ke liye career recommender ----
def recommend_careers(resume_skills, top_k=3, gap_n=8):
    x = cv_s.transform([sorted(set(resume_skills))])
    p = model_s.predict_proba(x)[0]
    for i in np.argsort(-p)[:top_k]:
        fam = model_s.classes_[i]
        gap = family_gap(resume_skills, fam, top_n=25)
        gap = gap[~gap["skill"].isin(GENERIC_SKILLS)].head(gap_n)
        learn = ", ".join(f"{s} ({pc}%)" for s, pc in zip(gap["skill"], gap["pct_of_family_listings"]))
        print(f"\n{fam}  | model probability {p[i]:.2f}")
        print("  skills to learn (% of that family's listings):", learn)

recommend_careers(resume_skills)

# ---- Model D se ambiguous postings ka family guess ----
amb = df[df["family"].eq(AMBIG)].copy()
pa = model_d.predict_proba(X_d(amb))
amb["pred_family"] = model_d.classes_[pa.argmax(1)]
amb["pred_conf"] = pa.max(1)
print("\nAmbiguous postings:", len(amb))
print(amb["pred_family"].value_counts().to_string())
print("share with confidence >= 0.6:", round((amb["pred_conf"] >= 0.6).mean(), 3))
print(amb.loc[amb["pred_conf"] >= 0.6, ["title", "pred_family", "pred_conf"]]
        .sample(12, random_state=1).round(2).to_string(index=False))

           model  top1_acc  top3_acc  macro_F1
  S: skills only     0.795     0.939     0.711
D: skills + desc     0.823     0.960     0.749

Data Scientist / ML-AI  | model probability 0.67
  skills to learn (% of that family's listings): natural language processing (25.3%), tensorflow (16.7%), generative ai (15.0%), deep learning (13.6%), pytorch (11.8%), image processing (10.1%), large language models (8.9%), chatbot (8.6%)

Software Developer  | model probability 0.33
  skills to learn (% of that family's listings): css (26.2%), java (23.6%), c# (16.8%), spring boot (13.2%), microservices (12.7%), javascript (14.2%), react.js (10.7%), hibernate (8.1%)

Data Engineer  | model probability 0.00
  skills to learn (% of that family's listings): pyspark (26.7%), scala (25.8%), data engineering (21.7%), hive (21.9%), etl (20.4%), hadoop (17.2%), spark (16.7%), airflow (14.9%)

Ambiguous postings: 1513
pred_family
Software Developer            1184
IT Support / Sys & Network     148
ERP / 

In [ ]:
prior = tr["family"].value_counts(normalize=True)

def recommend_careers_v2(resume_skills, top_k=3, gap_n=8, min_prob=0.05):
    known = [s for s in set(resume_skills) if s in cv_s.vocabulary_]
    x = cv_s.transform([sorted(known)])
    p = model_s.predict_proba(x)[0]
    print(f"Skills used by model: {len(known)} of {len(set(resume_skills))}")
    shown = 0
    for i in np.argsort(-p):
        if shown >= top_k or p[i] < min_prob:
            break
        fam = model_s.classes_[i]
        gap = family_gap(resume_skills, fam, top_n=25)
        gap = gap[~gap["skill"].isin(GENERIC_SKILLS)].head(gap_n)
        learn = ", ".join(f"{s} ({pc}%)" for s, pc in zip(gap["skill"], gap["pct_of_family_listings"]))
        print(f"  {fam} | prob {p[i]:.2f} | family is {prior[fam]*100:.1f}% of listings | lift x{p[i]/prior[fam]:.1f}")
        print(f"     learn: {learn}")
        shown += 1

PERSONAS = {
    "Backend fresher":  ["java", "spring boot", "sql", "mysql", "git", "rest apis", "microservices"],
    "Tester":           ["selenium", "manual testing", "automation testing", "java", "sql", "jira"],
    "Data engineer":    ["python", "sql", "pyspark", "airflow", "etl", "hive", "amazon web services"],
    "Security":         ["cyber security", "network security", "siem", "owasp", "linux"],
    "Web frontend":     ["javascript", "react.js", "html", "css", "typescript"],
}
for name, sk in PERSONAS.items():
    print(f"\n=== {name}")
    recommend_careers_v2(sk)


=== Backend fresher
Skills used by model: 7 of 7
  Software Developer | prob 0.96 | family is 37.7% of listings | lift x2.5
     learn: css (26.2%), c# (16.8%), javascript (14.2%), python (16.2%), react.js (10.7%), hibernate (8.1%), angular (8.0%), spring (7.2%)

=== Tester
Skills used by model: 6 of 6
  QA / Testing | prob 0.86 | family is 5.6% of listings | lift x15.4
     learn: software testing (31.8%), api testing (13.2%), quality assurance (13.2%), test cases (11.9%), regression testing (11.5%), performance testing (8.8%), automation framework (6.5%), python (14.9%)
  Software Developer | prob 0.09 | family is 37.7% of listings | lift x0.2
     learn: css (26.2%), c# (16.8%), spring boot (13.2%), microservices (12.7%), javascript (14.2%), python (16.2%), react.js (10.7%), hibernate (8.1%)

=== Data engineer
Skills used by model: 7 of 7
  Data Engineer | prob 0.97 | family is 6.5% of listings | lift x15.0
     learn: scala (25.8%), data engineering (21.7%), hadoop (17.2%), spark 

In [ ]:
import re
GENERIC_WORDS = {"development", "developer", "programming", "engineering", "framework", "skills", "application", "software"}
_tok = lambda s: set(re.findall(r"[a-z0-9#+.]+", s.lower()))

def _redundant(skill, resume_tok_list):
    tg = _tok(skill); core = tg - GENERIC_WORDS
    return any(tg <= tr or (core and core <= tr) for tr in resume_tok_list)

def personalized_gap(resume_skills, fam, k=200, gap_n=8):
    sub = it_df[it_df["family"].eq(fam)]
    rs = set(resume_skills)
    rs_tok = [_tok(s) for s in rs]
    known = [s for s in rs if s in cv_s.vocabulary_]
    sims = cosine_similarity(cv_s.transform([sorted(known)]), cv_s.transform(sub["skills_list"]))[0]
    nn = sub.iloc[np.argsort(-sims)[:k]]
    c = Counter(s for sk in nn["skills_list"] for s in set(sk))
    rows = [(s, round(100 * v / len(nn), 1)) for s, v in c.items()
            if s not in rs and s not in GENERIC_SKILLS and s in cv_s.vocabulary_
            and len(s) >= 3 and not _redundant(s, rs_tok)]
    return (pd.DataFrame(rows, columns=["skill", "pct_of_similar_listings"])
              .sort_values("pct_of_similar_listings", ascending=False).head(gap_n).reset_index(drop=True))

def roadmap_for(resume, missing, fam):
    """build_learning_roadmap reads the global `resume_skills`; swap it temporarily."""
    global resume_skills
    saved = resume_skills
    resume_skills = list(resume)
    try:
        return build_learning_roadmap(missing, target_family=fam)
    finally:
        resume_skills = saved

def career_report(resume, top_k=3, min_prob=0.05, gap_n=8, roadmap_for_top=True):
    known = [s for s in set(resume) if s in cv_s.vocabulary_]
    p = model_s.predict_proba(cv_s.transform([sorted(known)]))[0]
    print(f"Skills recognised: {len(known)} of {len(set(resume))}")
    shown = 0
    for i in np.argsort(-p):
        if shown >= top_k or p[i] < min_prob:
            break
        fam = model_s.classes_[i]
        gap = personalized_gap(resume, fam, gap_n=gap_n)
        print(f"\n#{shown+1} {fam} | prob {p[i]:.2f} | lift x{p[i]/prior[fam]:.1f}")
        print("   learn next:", ", ".join(f"{s} ({v}%)" for s, v in zip(gap['skill'], gap['pct_of_similar_listings'])))
        if shown == 0 and roadmap_for_top and len(gap):
            rm = roadmap_for(resume, gap["skill"].tolist(), fam)
            print("   roadmap (order):", " -> ".join(rm["skill_to_learn"].tolist()))
        shown += 1

for name, sk in PERSONAS.items():
    print(f"\n==================== {name}")
    career_report(sk)


==================== Backend fresher
Skills recognised: 7 of 7

#1 Software Developer | prob 0.96 | lift x2.5
   learn next: react.js (32.5%), angular (32.0%), hibernate (20.0%), full stack (15.5%), amazon web services (9.5%), stack (9.0%), bootstrap (8.0%), kafka (7.0%)
   roadmap (order): react.js -> hibernate -> angular -> full stack -> bootstrap -> stack -> kafka -> linux -> amazon web services

==================== Tester
Skills recognised: 6 of 6

#1 QA / Testing | prob 0.86 | lift x15.4
   learn next: api testing (24.5%), continuous integration/continuous delivery (14.5%), python (13.5%), regression testing (11.5%), cucumber (11.0%), api (10.5%), appium (9.0%), quality assurance (8.0%)
   roadmap (order): python -> quality assurance -> api testing -> regression testing -> continuous integration/continuous delivery -> appium -> api -> cucumber

#2 Software Developer | prob 0.09 | lift x0.2
   learn next: spring boot (42.0%), microservices (38.0%), angular (27.0%), react.js (24.5

In [ ]:
_pg_old = personalized_gap
def personalized_gap(resume_skills, fam, k=200, gap_n=8):
    g = _pg_old(resume_skills, fam, k=k, gap_n=gap_n * 2)
    toks = [(_tok(s), s) for s in g["skill"]]
    keep = [s for t, s in toks if not any(t < t2 for t2, s2 in toks if s2 != s)]
    return g[g["skill"].isin(keep)].head(gap_n).reset_index(drop=True)

# min_prob ko 0.10 karne ke liye career_report ko aise chalao:
career_report(PERSONAS["Backend fresher"], min_prob=0.10)

Skills recognised: 7 of 7

#1 Software Developer | prob 0.96 | lift x2.5
   learn next: react.js (32.5%), angular (32.0%), hibernate (20.0%), full stack (15.5%), amazon web services (9.5%), bootstrap (8.0%), kafka (7.0%), docker (5.0%)
   roadmap (order): react.js -> hibernate -> angular -> full stack -> bootstrap -> kafka -> linux -> amazon web services -> docker


In [ ]:
def gap_hit_eval(n_eval=1000, k=200, topn=8, seed=0):
    rng = np.random.RandomState(seed)
    cand = te[te["skills_list"].map(len) >= 5]
    cand = cand.sample(min(n_eval, len(cand)), random_state=seed)
    cache, pop = {}, {}
    for fam in cand["family"].unique():
        sub = tr[tr["family"].eq(fam)]
        cache[fam] = (sub, cv_s.transform(sub["skills_list"]))
        pop[fam] = Counter(s for sk in sub["skills_list"] for s in set(sk))
    glob = Counter(s for sk in tr["skills_list"] for s in set(sk))
    res = {"personalized (kNN)": [], "family popularity": [], "global popularity": []}
    for _, row in cand.iterrows():
        skills = [s for s in row["skills_list"] if s in cv_s.vocabulary_]
        if len(skills) < 5:
            continue
        perm = rng.permutation(len(skills)); cut = int(round(0.6 * len(skills)))
        shown = [skills[i] for i in perm[:cut]]
        hidden = {skills[i] for i in perm[cut:]} - GENERIC_SKILLS
        if not hidden:
            continue
        sub, Xsub = cache[row["family"]]
        sims = cosine_similarity(cv_s.transform([sorted(shown)]), Xsub)[0]
        nn = sub.iloc[np.argsort(-sims)[:k]]
        c = Counter(s for sk in nn["skills_list"] for s in set(sk))
        for name, counter in [("personalized (kNN)", c), ("family popularity", pop[row["family"]]),
                              ("global popularity", glob)]:
            rec = [s for s, _ in counter.most_common(80)
                   if s not in shown and s not in GENERIC_SKILLS and s in cv_s.vocabulary_][:topn]
            hit = len(set(rec) & hidden)
            res[name].append((hit / len(hidden), int(hit > 0)))
    out = pd.DataFrame({n: {"recall_of_hidden_skills": np.mean([r for r, _ in v]),
                            "hit_rate_any": np.mean([h for _, h in v]),
                            "n_listings": len(v)} for n, v in res.items()}).T
    return out.round(3)

print(gap_hit_eval())

                    recall_of_hidden_skills  hit_rate_any  n_listings
personalized (kNN)                    0.338         0.600       952.0
family popularity                     0.227         0.455       952.0
global popularity                     0.107         0.251       952.0


In [ ]:
for sd in [1, 2]:
    print(f"\nseed = {sd}")
    print(gap_hit_eval(seed=sd))


seed = 1
                    recall_of_hidden_skills  hit_rate_any  n_listings
personalized (kNN)                    0.363         0.615       948.0
family popularity                     0.246         0.474       948.0
global popularity                     0.111         0.264       948.0

seed = 2
                    recall_of_hidden_skills  hit_rate_any  n_listings
personalized (kNN)                    0.350         0.607       956.0
family popularity                     0.240         0.463       956.0
global popularity                     0.113         0.263       956.0


In [ ]:
real_fams = sorted(f for f in df["family"].unique() if f not in ("Other / Non-IT or Unclear", AMBIG))
parts = []
for fam in real_fams:
    g = df[df["family"].eq(fam)]
    parts.append(g.sample(n=min(8, len(g)), random_state=7).assign(sheet="A_rule_label", label_to_check=fam))
parts.append(amb.sample(n=min(30, len(amb)), random_state=7)
               .assign(sheet="B_ambiguous", label_to_check=lambda x: x["pred_family"]))
parts.append(df[df["family"].eq("Other / Non-IT or Unclear")].sample(n=30, random_state=7)
               .assign(sheet="C_other", label_to_check="Other / Non-IT or Unclear"))

val = pd.concat(parts).sample(frac=1, random_state=7).reset_index(drop=True)
val["row_id"] = np.arange(1, len(val) + 1)
val["top_skills"] = val["skills_list"].map(lambda xs: ", ".join(xs[:8]))
val["desc_snippet"] = val["description_clean"].str[:250]

blind = val[["row_id", "title", "companyName", "top_skills", "desc_snippet"]].copy()
blind["your_family"] = ""
key = val[["row_id", "sheet", "jobId", "label_to_check"]]

out_dir = Path(DRIVE_BASE) / "processed"
blind.to_csv(out_dir / "validation_BLIND_to_label.csv", index=False)
key.to_csv(out_dir / "validation_KEY_do_not_open.csv", index=False)
print("rows:", len(blind))
print("Allowed values for your_family:")
for f in real_fams + ["IT - other/unclear", "Not IT"]:
    print("  ", f)

rows: 180
Allowed values for your_family:
   Business / Systems Analyst
   Cybersecurity
   Data / BI Analyst
   Data Engineer
   Data Scientist / ML-AI
   Database Admin
   DevOps / Cloud
   ERP / CRM Consultant
   Frontend / Web
   IT Architect
   IT Support / Sys & Network
   Mobile Developer
   QA / Testing
   Software Developer
   Tech Project / Product
   IT - other/unclear
   Not IT


## 13. Role-family demand analysis

This is descriptive analysis of the dataset, not a live vacancy count. The dataset is a historical snapshot and may contain stale or duplicate listings.

In [ ]:
it_jobs = df[df["family"].ne("Other / Non-IT or Unclear")].copy()
display(it_jobs["family"].value_counts().rename_axis("IT job family").to_frame("listing_count"))
display(
    it_jobs.groupby("family")
    .agg(listings=("jobId", "count"),
         median_min_experience=("experience_min", "median"),
         median_max_experience=("experience_max", "median"),
         companies=("companyName", "nunique"))
    .sort_values("listings", ascending=False)
)

## 14. Manual validation sheet

A sample of job titles is exported for you to label whether the assigned family is correct. This is the right way to estimate rule-based classification quality; don't report a guessed accuracy.

In [ ]:
validation_parts = []
for fam, group in df.groupby("family", dropna=False):
    n = min(25, len(group))
    if n:
        validation_parts.append(group.sample(n=n, random_state=SEED))
validation = pd.concat(validation_parts, ignore_index=True).drop_duplicates("jobId")
validation = validation[["jobId", "title", "family", "companyName", "location"]].copy()
validation["is_correct"] = ""      # Fill Y / N
validation["correct_family"] = ""  # If N, enter the right family
VALIDATION_PATH = Path("job_family_validation.csv")
validation.to_csv(VALIDATION_PATH, index=False)
print("Saved:", VALIDATION_PATH.resolve(), "| rows:", len(validation))
display(validation.head(15))

## 15. Basic system checks and export processed data

These checks catch common failures but are not a substitute for manual evaluation. We export processed jobs and the recommendation results for reproducibility.

In [ ]:
assert len(df) > 0, "No rows after cleaning."
assert df["jobId"].notna().all(), "Some job IDs are missing."
assert df["family"].notna().all(), "Some rows have no family label."
assert all(isinstance(x, list) for x in df["skills_list"].head(100)), "Skills should be lists."
if not recommendations.empty:
    assert recommendations["recommendation_score"].between(0, 1).all(), "Scores should be between 0 and 1."
    assert recommendations["recommendation_score"].is_monotonic_decreasing, "Recommendations should be sorted."

OUT_DIR = Path("job_recommendation_outputs")
OUT_DIR.mkdir(exist_ok=True)
df.to_pickle(OUT_DIR / "jobs_processed.pkl")
df.drop(columns=["skills_list"], errors="ignore").to_csv(OUT_DIR / "jobs_processed.csv", index=False)
if not recommendations.empty:
    recommendations.to_csv(OUT_DIR / "recommendations.csv", index=False)
if not roadmap.empty:
    roadmap.to_csv(OUT_DIR / "learning_roadmap.csv", index=False)
family_counts.to_csv(OUT_DIR / "family_counts.csv")
print("Exports saved in:", OUT_DIR.resolve())

## 16. How to report results honestly

After you manually label the validation sheet:
- Calculate precision/recall per family only after mapping your labels back to the sampled rows.
- For recommendation evaluation, create several test resumes and manually judge whether the top-10 roles are relevant. Report Precision@5 / Precision@10 only for examples you actually label.
- Compare the ranking with a simple baseline (e.g., skill overlap only) to see whether TF-IDF adds value.
- Tune weights only on your validation examples; don't describe arbitrary weights as scientifically optimal.
- Do not claim the system predicts hiring probability, salary, or job success.

### Suggested next improvements
1. Add more aliases and skill phrases after reviewing false positives / false negatives.
2. Add a small labelled dataset for role-family classification.
3. Compare TF-IDF with a sentence-transformer embedding model and measure top-k relevance on labelled resumes.
4. Add resume sections (skills/projects/experience) with different weights.
5. Add tests for common edge cases: empty resume, scanned PDF, resume with no detected skills, job with no skill tags, and unknown city.
6. Only after the notebook works reliably, build a Streamlit UI.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/Job recommendation nd skill gap analysis"
df = pd.read_pickle(Path(DRIVE_BASE) / "processed" / "jobs_checkpoint_v2.pkl")
AMBIG = "IT Services - Ambiguous Role"
real_fams = sorted(f for f in df["family"].unique() if f not in ("Other / Non-IT or Unclear", AMBIG))
print("rows:", len(df), "| families:", len(real_fams))

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (float(round(c - h, 3)), float(round(c + h, 3)))

lab = pd.read_csv(Path(DRIVE_BASE) / "processed" / "validation_labeled.csv")
key = pd.read_csv(Path(DRIVE_BASE) / "processed" / "validation_KEY_do_not_open.csv")
m = lab[["row_id", "your_family"]].merge(key, on="row_id")
m["your_family"] = m["your_family"].fillna("").astype(str).str.strip()

allowed = set(real_fams) | {"IT - other/unclear", "Not IT"}
bad = m[~m["your_family"].isin(allowed)]
print("rows:", len(m), "| empty or unrecognised labels:", len(bad))
if len(bad):
    print(bad[["row_id", "your_family"]].head(10).to_string(index=False))

m = m[m["your_family"].isin(allowed)]

# A) rule-based family labels
a = m[m["sheet"].eq("A_rule_label")]
k, n = int((a["your_family"] == a["label_to_check"]).sum()), len(a)
print(f"\nA) Rule labels: exact agreement {k}/{n} = {k/n:.3f}  95% CI {wilson(k, n)}")
kn = int((a["your_family"] == "Not IT").sum())
print(f"   labelled 'Not IT' by you: {kn}/{n} = {kn/n:.3f}  (false-positive IT rate)")
print((a.assign(agree=a["your_family"] == a["label_to_check"])
        .groupby("label_to_check")["agree"].agg(["sum", "count"])
        .rename(columns={"sum": "agree", "count": "n"}).to_string()))

# B) Model D on ambiguous postings
b = m[m["sheet"].eq("B_ambiguous")]
k, n = int((b["your_family"] == b["label_to_check"]).sum()), len(b)
print(f"\nB) Model D on ambiguous postings: agreement {k}/{n} = {k/n:.3f}  95% CI {wilson(k, n)}")
print("   your labels:", b["your_family"].value_counts().to_dict())

# C) IT jobs hiding in 'Other'
c = m[m["sheet"].eq("C_other")]
ci = c["your_family"].isin(real_fams)
k, n = int(ci.sum()), len(c)
n_other = int(df["family"].eq("Other / Non-IT or Unclear").sum())
lo, hi = wilson(k, n)
print(f"\nC) 'Other' bucket: you labelled {k}/{n} as a real IT family = {k/n:.3f}  95% CI ({lo}, {hi})")
print(f"   => roughly {int(k/n*n_other)} IT jobs missed out of {n_other} (CI {int(lo*n_other)} to {int(hi*n_other)})")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rows: 93729 | families: 15
rows: 180 | empty or unrecognised labels: 0

A) Rule labels: exact agreement 102/120 = 0.850  95% CI (0.775, 0.903)
   labelled 'Not IT' by you: 2/120 = 0.017  (false-positive IT rate)
                            agree  n
label_to_check                      
Business / Systems Analyst      6  8
Cybersecurity                   8  8
Data / BI Analyst               7  8
Data Engineer                   8  8
Data Scientist / ML-AI          8  8
Database Admin                  7  8
DevOps / Cloud                  5  8
ERP / CRM Consultant            7  8
Frontend / Web                  6  8
IT Architect                    6  8
IT Support / Sys & Network      7  8
Mobile Developer                7  8
QA / Testing                    7  8
Software Developer              6  8
Tech Project / Product          7  8

B) Model D on ambiguous posti

In [ ]:
full = lab.merge(key, on="row_id")
full["your_family"] = full["your_family"].fillna("").astype(str).str.strip()

print("=== C) Jobs jo 'Other' mein thi par tumne IT family di ===")
c7 = full[(full["sheet"] == "C_other") & full["your_family"].isin(real_fams)]
print(c7[["title", "top_skills", "your_family"]].to_string(index=False))

print("\n=== A) Rule label vs tumhara label (disagreements) ===")
a_dis = full[(full["sheet"] == "A_rule_label") & (full["your_family"] != full["label_to_check"])]
print(a_dis[["title", "label_to_check", "your_family"]].to_string(index=False))

print("\n=== B) Ambiguous: Model D vs tum ===")
b_all = full[full["sheet"] == "B_ambiguous"]
print(b_all[["title", "label_to_check", "your_family"]].to_string(index=False))

=== C) Jobs jo 'Other' mein thi par tumne IT family di ===
                                                   title                                                                                                                                                    top_skills                your_family
                       Jr. Design Automation Engineer(5)                                                                                 cad customization, solid works, inventor, vb, sql database, c++, cad, autocad         Software Developer
                                 SSE - Python automation                                                                    microservices, network automation, python, spring boot, opennms, nagios, redhat linux, ssh             DevOps / Cloud
                                       Core Engineer 511                                                                      3dx, java fullstack, change management, cost management, bom management, core, java, bom 

In [ ]:
from sklearn.metrics import roc_auc_score

OTHER = "Other / Non-IT or Unclear"

def norm_id(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.lstrip("0")

oth = df[df["family"].eq(OTHER)]

d2 = df[["jobId", "p_it_skills"]].copy()
d2["jid"] = norm_id(d2["jobId"])
c30 = full[full["sheet"] == "C_other"][["jobId", "title", "your_family"]].copy()
c30["jid"] = norm_id(c30["jobId"])

dev = c30.merge(d2[["jid", "p_it_skills"]], on="jid")
print("matched rows:", len(dev), "of", len(c30))

dev["human_it"] = dev["your_family"].isin(real_fams).astype(int)
print("AUC on the labelled rows:", round(roc_auc_score(dev["human_it"], dev["p_it_skills"]), 3),
      "| n =", len(dev), "| IT =", int(dev["human_it"].sum()))
print(dev.sort_values("p_it_skills", ascending=False)[["title", "human_it", "p_it_skills"]]
        .round(2).to_string(index=False))

print("\nTop 25 'Other' rows by p_it (kya ye IT jaisi lagti hain?):")
top = oth.sort_values("p_it_skills", ascending=False).head(25)
print(top.assign(sk=top["skills_list"].map(lambda x: ", ".join(x[:5])))[["title", "sk", "p_it_skills"]]
        .round(2).to_string(index=False))

matched rows: 30 of 30
AUC on the labelled rows: 0.901 | n = 30 | IT = 7
                                                               title  human_it  p_it_skills
                                                      Senior Analyst         1         0.69
                                       Email Specialist || Bengaluru         0         0.47
                                             SSE - Python automation         1         0.47
                                              Java + Spark Developer         1         0.39
                                  Computer Hardware ( FTCP ) Trainer         1         0.24
            Performance Testing / Performance Engineering Specialist         1         0.16
                                                   Core Engineer 511         1         0.15
                                                    Supplier quality         0         0.15
PTP Specialist Accounts Payable, Invoice & Payment Processing Vendor         0         0.05
       

In [ ]:
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

# scores ko save kar lo, taaki session disconnect ho to dobara train na karna pade
df[["jobId", "p_it_skills"]].to_pickle(Path(DRIVE_BASE) / "processed" / "p_it_skills_v1.pkl")

OTHER = "Other / Non-IT or Unclear"
def norm_id(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.lstrip("0")

oth = df[df["family"].eq(OTHER)].copy()
oth["jid"] = norm_id(oth["jobId"])
oth = oth[~oth["jid"].isin(set(norm_id(key["jobId"])))]          # pehle dekhi hui 180 rows hata do

edges = [0, 0.05, 0.15, 0.5, 0.9, 1.0001]
names = ["p<0.05", "0.05-0.15", "0.15-0.5", "0.5-0.9", ">=0.9"]
oth["bin"] = pd.cut(oth["p_it_skills"], edges, labels=names, right=False)

N_PER_BIN = 16
parts, sizes = [], {}
for b in names:
    g = oth[oth["bin"] == b]
    sizes[b] = len(g)
    parts.append(g.sample(n=min(N_PER_BIN, len(g)), random_state=11))
s2 = pd.concat(parts).sample(frac=1, random_state=11).reset_index(drop=True)
s2["row_id"] = np.arange(1, len(s2) + 1)
s2["top_skills"] = s2["skills_list"].map(lambda xs: ", ".join(xs[:8]))
s2["desc_snippet"] = s2["description_clean"].str[:250]

blind = s2[["row_id", "title", "companyName", "top_skills", "desc_snippet"]].copy()
blind["your_family"] = ""
key2 = s2[["row_id", "jobId", "bin", "p_it_skills"]]

out = Path(DRIVE_BASE) / "processed"
xlsx = out / "validation2_to_label.xlsx"
blind.to_excel(xlsx, index=False, sheet_name="label")
key2.to_csv(out / "validation2_KEY_do_not_open.csv", index=False)
pd.Series(sizes, name="N_in_bin").to_csv(out / "validation2_bin_sizes.csv")

opts = list(real_fams) + ["IT - other/unclear", "Not IT"]
wb = load_workbook(xlsx)
ws = wb["label"]
o = wb.create_sheet("options")
for v in opts:
    o.append([v])
for k, w in {"A": 8, "B": 38, "C": 24, "D": 50, "E": 70, "F": 30}.items():
    ws.column_dimensions[k].width = w
for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
    for c in row:
        c.alignment = Alignment(wrap_text=True, vertical="top")
for c in ws[1]:
    c.font = Font(bold=True, color="FFFFFF"); c.fill = PatternFill("solid", fgColor="305496")
for r in range(2, ws.max_row + 1):
    ws.cell(r, 6).fill = PatternFill("solid", fgColor="FFF2CC")
ws.freeze_panes = "B2"
dv = DataValidation(type="list", formula1=f"=options!$A$1:$A${len(opts)}", allow_blank=True,
                    showErrorMessage=True, errorTitle="Invalid", error="Dropdown se value chuno")
ws.add_data_validation(dv); dv.add(f"F2:F{ws.max_row}")
wb.save(xlsx)

print("rows to label:", len(blind))
print("rows per probability bin in ALL 'Other' (excl. earlier 180):")
print(pd.Series(sizes).to_string())
print("saved:", xlsx)

rows to label: 80
rows per probability bin in ALL 'Other' (excl. earlier 180):
p<0.05       51538
0.05-0.15     8558
0.15-0.5      6229
0.5-0.9       3819
>=0.9          641
saved: /content/drive/MyDrive/Job recommendation nd skill gap analysis/processed/validation2_to_label.xlsx


In [ ]:
import os
p = "/content/drive/MyDrive/Job recommendation nd skill gap analysis/processed"
print([f for f in os.listdir(p) if "validation2" in f])

['validation2_to_label.xlsx', 'validation2_KEY_do_not_open.csv', 'validation2_bin_sizes.csv', 'validation2_labeled.csv']


In [ ]:
p = pd.read_pickle(Path(DRIVE_BASE) / "processed" / "p_it_skills_v1.pkl")
df["p_it_skills"] = df["jobId"].map(p.set_index("jobId")["p_it_skills"])
print("rows:", len(df), "| scores attached:", df["p_it_skills"].notna().mean().round(3))

rows: 93729 | scores attached: 1.0


In [ ]:
OTHER = "Other / Non-IT or Unclear"
edges = [0, 0.05, 0.15, 0.5, 0.9, 1.0001]
names = ["p<0.05", "0.05-0.15", "0.15-0.5", "0.5-0.9", ">=0.9"]

lab2 = pd.read_csv(Path(DRIVE_BASE) / "processed" / "validation2_labeled.csv")
key2 = pd.read_csv(Path(DRIVE_BASE) / "processed" / "validation2_KEY_do_not_open.csv")
m2 = lab2[["row_id", "your_family"]].merge(key2, on="row_id")
m2["your_family"] = m2["your_family"].fillna("").astype(str).str.strip()
allowed = set(real_fams) | {"IT - other/unclear", "Not IT"}
print("rows:", len(m2), "| empty/invalid labels:", int((~m2["your_family"].isin(allowed)).sum()))
m2["is_it"] = m2["your_family"].isin(real_fams).astype(int)
m2["bin"] = pd.cut(m2["p_it_skills"], edges, labels=names, right=False)
samples = m2[["bin", "is_it"]]

# pehle wali 30 random 'Other' rows bhi pool kar lo (ye bhi uniform sample hain)
if "dev" in globals():
    d = dev[["p_it_skills", "human_it"]].rename(columns={"human_it": "is_it"})
    d["bin"] = pd.cut(d["p_it_skills"], edges, labels=names, right=False)
    samples = pd.concat([samples, d[["bin", "is_it"]]])
    print("pooled the 30 earlier random rows as well")

oth_all = df[df["family"].eq(OTHER)]
N_bin = pd.cut(oth_all["p_it_skills"], edges, labels=names, right=False).value_counts().reindex(names)

def wilson(k, n, z=1.96):
    p = k / n; dd = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / dd
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / dd
    return round(c - h, 3), round(c + h, 3)

rows = []
for b in names:
    s = samples[samples["bin"] == b]
    n, k = len(s), int(s["is_it"].sum())
    rows.append({"bin": b, "N_in_bin": int(N_bin[b]), "n_labelled": n, "IT_found": k,
                 "IT_rate": round(k / n, 3), "rate_95CI": wilson(k, n),
                 "est_IT_jobs": int(round(k / n * N_bin[b]))})
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))

# Total chhooti hui IT jobs: Monte Carlo (Beta posterior per bin)
rng = np.random.default_rng(0)
tot = np.zeros(20000)
for r in rows:
    tot += rng.beta(r["IT_found"] + 0.5, r["n_labelled"] - r["IT_found"] + 0.5, 20000) * r["N_in_bin"]
lo, med, hi = np.percentile(tot, [2.5, 50, 97.5])
missed = sum(r["est_IT_jobs"] for r in rows)
print(f"\nEstimated IT jobs hiding in 'Other': ~{missed}  (95% interval {int(lo)} to {int(hi)})")
n_rule_it = int(df["family"].ne(OTHER).sum())
print(f"Rule-IT listings: {n_rule_it}  => rough recall of title rules ~ {n_rule_it/(n_rule_it+missed):.2f}"
      "  (rule false positives ignore kiye)")

print("\nDetector at different cut-offs (point estimates):")
cuts = [("p>=0.05", names[1:]), ("p>=0.15", names[2:]), ("p>=0.5", names[3:]), ("p>=0.9", names[4:])]
out = []
for nm, bl in cuts:
    flagged = sum(r["N_in_bin"] for r in rows if r["bin"] in bl)
    hit = sum(r["est_IT_jobs"] for r in rows if r["bin"] in bl)
    out.append({"cutoff": nm, "flagged": flagged, "est_true_IT": hit,
                "precision": round(hit / flagged, 3), "recall_of_missed": round(hit / missed, 3)})
print(pd.DataFrame(out).to_string(index=False))

rows: 80 | empty/invalid labels: 0
pooled the 30 earlier random rows as well
      bin  N_in_bin  n_labelled  IT_found  IT_rate      rate_95CI  est_IT_jobs
   p<0.05     51560          38         2    0.053 (0.015, 0.173)         2714
0.05-0.15      8559          17         5    0.294 (0.133, 0.531)         2517
 0.15-0.5      6235          22        17    0.773 (0.566, 0.899)         4818
  0.5-0.9      3820          17        16    0.941   (0.73, 0.99)         3595
    >=0.9       641          16        16    1.000   (0.806, 1.0)          641

Estimated IT jobs hiding in 'Other': ~14285  (95% interval 11165 to 20098)
Rule-IT listings: 22914  => rough recall of title rules ~ 0.62  (rule false positives ignore kiye)

Detector at different cut-offs (point estimates):
 cutoff  flagged  est_true_IT  precision  recall_of_missed
p>=0.05    19255        11571      0.601             0.810
p>=0.15    10696         9054      0.846             0.634
 p>=0.5     4461         4236      0.950      

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd
from pathlib import Path
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression

DRIVE_BASE = "/content/drive/MyDrive/Job recommendation nd skill gap analysis"
df = pd.read_pickle(Path(DRIVE_BASE) / "processed" / "jobs_checkpoint_v2.pkl")
print("loaded:", df.shape)

OTHER = "Other / Non-IT or Unclear"; AMBIG = "IT Services - Ambiguous Role"
if "p_it_skills" not in df.columns:
    p = pd.read_pickle(Path(DRIVE_BASE) / "processed" / "p_it_skills_v1.pkl")
    df = df.merge(p, on="jobId", how="left")
print("rows without detector score:", int(df["p_it_skills"].isna().sum()))
real_fams = sorted(f for f in df["family"].unique() if f not in (OTHER, AMBIG))

# Family model: SIRF rule-labelled IT rows pe train (ambiguous aur predicted rows training mein nahi)
train = df[df["family"].isin(real_fams)]
cv_f = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3)
tf_f = TfidfVectorizer(max_features=20000, min_df=5, sublinear_tf=True, ngram_range=(1, 2), stop_words="english")
Xd = hstack([cv_f.fit_transform(train["skills_list"]), tf_f.fit_transform(train["description_clean"])]).tocsr()
model_fam = LogisticRegression(max_iter=1000).fit(Xd, train["family"])

def predict_family(frame):
    X = hstack([cv_f.transform(frame["skills_list"]), tf_f.transform(frame["description_clean"])]).tocsr()
    P = model_fam.predict_proba(X)
    return model_fam.classes_[P.argmax(1)], P.max(1)

ext = df.copy()
ext["family_final"] = ext["family"]
ext["family_source"] = np.where(ext["family"].isin(real_fams), "rule",
                        np.where(ext["family"].eq(AMBIG), "ambiguous", "none"))
ext["pred_conf"] = np.nan
ext["it_tier"] = "non_IT_or_unknown"
ext.loc[ext["family"].isin(real_fams), "it_tier"] = "core_rule"
ext.loc[ext["family"].eq(AMBIG), "it_tier"] = "core_ambiguous"

cand = ext[ext["family"].eq(OTHER) & (ext["p_it_skills"] >= 0.15)]
fam_pred, conf = predict_family(cand)
ext.loc[cand.index, "family_final"] = fam_pred
ext.loc[cand.index, "family_source"] = "predicted"
ext.loc[cand.index, "pred_conf"] = conf
ext.loc[cand.index, "it_tier"] = np.where(cand["p_it_skills"] >= 0.5, "extended_high", "extended_mid")

print(ext["it_tier"].value_counts().to_string())
print("\nfamily_final counts (core + predicted):")
print(ext[ext["family_source"].isin(["rule", "predicted"])]["family_final"].value_counts().to_string())
ext.to_pickle(Path(DRIVE_BASE) / "processed" / "jobs_extended_v1.pkl")
print("\nsaved jobs_extended_v1.pkl:", ext.shape)

Mounted at /content/drive
loaded: (93729, 28)
rows without detector score: 0
it_tier
non_IT_or_unknown    60119
core_rule            21401
extended_mid          6235
extended_high         4461
core_ambiguous        1513

family_final counts (core + predicted):
family_final
Software Developer            11417
ERP / CRM Consultant           5858
QA / Testing                   2205
Data Engineer                  1885
IT Support / Sys & Network     1797
DevOps / Cloud                 1620
Data Scientist / ML-AI         1535
Frontend / Web                 1008
Data / BI Analyst               984
Business / Systems Analyst      876
Cybersecurity                   851
Tech Project / Product          743
IT Architect                    697
Mobile Developer                317
Database Admin                  304

saved jobs_extended_v1.pkl: (93729, 33)


run daily

In [1]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['ADZUNA_APP_ID'] = userdata.get('ADZUNA_APP_ID')
os.environ['ADZUNA_APP_KEY'] = userdata.get('ADZUNA_APP_KEY')
%run "/content/drive/MyDrive/Job recommendation nd skill gap analysis/adzuna_daily_collect.py"

Mounted at /content/drive
2026-11-16: 47 calls, 478 NEW postings, 3325 unique so far -> 2026-11-16.jsonl
Budget used: day 47/200, week 188/800, month 188/2000


In [2]:
import pandas as pd, json
from pathlib import Path

base = Path("/content/drive/MyDrive/Job recommendation nd skill gap analysis/adzuna/raw")
rows = [json.loads(l) for f in sorted(base.glob("*.jsonl")) for l in open(f, encoding="utf-8")]
a = pd.DataFrame(rows)
print(a.shape)
print(a.columns.tolist())

a["desc_words"] = a["description"].fillna("").str.split().str.len()
print("\nDescription words:")
print(a["desc_words"].describe().round(1).to_string())

a["company_name"] = a["company"].map(lambda c: c.get("display_name") if isinstance(c, dict) else None)
a["location_name"] = a["location"].map(lambda l: l.get("display_name") if isinstance(l, dict) else None)
a["category_label"] = a["category"].map(lambda c: c.get("label") if isinstance(c, dict) else None)

print("\nTop locations:\n", a["location_name"].value_counts().head(8).to_string())
print("\nTop categories:\n", a["category_label"].value_counts().head(8).to_string())
if "salary_min" in a:
    print("\nsalary present:", round(a["salary_min"].notna().mean(), 3))
print("\nPostings per query:\n", a.groupby("query").size().to_string())
print("\nRandom sample:")
print(a[["title", "company_name", "location_name"]].sample(10, random_state=1).to_string(index=False))

(3325, 19)
['id', 'created', 'adref', 'location', 'category', 'salary_is_predicted', 'company', 'contract_type', '__CLASS__', 'description', 'title', 'contract_time', 'redirect_url', 'query', 'collected_on', 'latitude', 'longitude', 'salary_min', 'salary_max']

Description words:
count    3325.0
mean       72.4
std         6.6
min        22.0
25%        68.0
50%        72.0
75%        76.0
max        94.0

Top locations:
 location_name
India                   1333
Bangalore, Karnataka     755
Hyderabad, Telangana     302
Pune, Maharashtra        207
Chennai, Tamil Nadu      154
Mumbai, Maharashtra       98
Noida, Ghaziabad          67
Kolkata, West Bengal      31

Top categories:
 category_label
IT Jobs                             2790
Engineering Jobs                     231
Accounting & Finance Jobs             90
Consultancy Jobs                      69
Scientific & QA Jobs                  43
Sales Jobs                            18
HR & Recruitment Jobs                 17
PR, Adve

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import sys, pandas as pd
BASE = "/content/drive/MyDrive/Job recommendation nd skill gap analysis"
sys.path.append(BASE)
from career_utils import *

ext = SkillExtractor(load_vocab(f"{BASE}/processed/skill_vocab_v1.pkl"))
live = load_adzuna(f"{BASE}/adzuna/raw", ext)
print(live.shape)
print(live["family"].value_counts().to_string())
print("\nIT by title rules:", int((live["family"] != OTHER).sum()), "of", len(live))
print("\nAdzuna category vs IT-by-rules:")
print(pd.crosstab(live["category"].fillna("?"), live["family"].ne(OTHER)).head(8))
print("\ntext-skills per posting:", live["n_skills_txt"].describe().round(1).to_dict())
print("share with >=3 skills:", round((live["n_skills_txt"] >= 3).mean(), 3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(3209, 19)
family
Other / Non-IT or Unclear       1291
Software Developer               716
ERP / CRM Consultant             232
DevOps / Cloud                   169
Data Scientist / ML-AI           143
Data Engineer                    140
QA / Testing                     119
Cybersecurity                     73
Business / Systems Analyst        61
Data / BI Analyst                 60
Tech Project / Product            47
Database Admin                    41
Mobile Developer                  39
Frontend / Web                    26
IT Architect                      24
IT Support / Sys & Network        21
IT Services - Ambiguous Role       7

IT by title rules: 1918 of 3209

Adzuna category vs IT-by-rules:
family                     False  True 
category                               
Accounting & Finance Jobs     56     34
Admin Jobs                     8      

In [4]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack

# --- 1) Adzuna category vs title rules (poori table)
print("Adzuna category vs IT-by-title-rules:")
ct = pd.crosstab(live["category"].fillna("?"), live["family"].ne(OTHER))
ct.columns = ["rules_say_Other", "rules_say_IT"]
ct["share_IT_by_rules"] = (ct["rules_say_IT"] / ct.sum(axis=1)).round(2)
print(ct.sort_values("rules_say_IT", ascending=False).to_string())

# --- 2) Preliminary shift test: 2025 snapshot pe train, live pe test (title-rule labels)
hist = pd.read_pickle(f"{BASE}/processed/jobs_checkpoint_v2.pkl")
core = hist[hist["family"].ne(OTHER) & hist["family"].ne(AMBIG)].copy()
core["skills_txt"] = core["description_clean"].map(ext.extract)        # dono periods ke liye same extractor
test = live[live["family"].ne(OTHER) & live["family"].ne(AMBIG)].copy()
print("\ntrain (2025 core):", len(core), "| live test rows:", len(test))

def fit_eval(use_skills):
    xa, xb = [], []
    tf = TfidfVectorizer(max_features=20000, min_df=5, sublinear_tf=True, ngram_range=(1, 2), stop_words="english")
    xa.append(tf.fit_transform(core["description_clean"].fillna(""))); xb.append(tf.transform(test["description_clean"].fillna("")))
    if use_skills:
        cv = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3)
        xa.append(cv.fit_transform(core["skills_txt"])); xb.append(cv.transform(test["skills_txt"]))
    m = LogisticRegression(max_iter=1000).fit(hstack(xa).tocsr(), core["family"])
    return m.predict(hstack(xb).tocsr())

maj = core["family"].value_counts().idxmax()
print(f"baseline (always '{maj}'): top1 {(test['family'] == maj).mean():.3f}")
for name, flag in [("description TF-IDF only", False), ("description TF-IDF + text skills", True)]:
    p = fit_eval(flag)
    print(f"{name:34s} top1 {(p == test['family']).mean():.3f}  macroF1 {f1_score(test['family'], p, average='macro'):.3f}")
print("\nPer-family report (last variant):")
print(classification_report(test["family"], p, digits=2, zero_division=0))

Adzuna category vs IT-by-title-rules:
                                  rules_say_Other  rules_say_IT  share_IT_by_rules
category                                                                          
IT Jobs                                       946          1750               0.65
Engineering Jobs                              154            69               0.31
Consultancy Jobs                               21            35               0.62
Accounting & Finance Jobs                      56            34               0.38
Scientific & QA Jobs                           25            17               0.40
PR, Advertising & Marketing Jobs               10             2               0.17
Manufacturing Jobs                              2             2               0.50
Healthcare & Nursing Jobs                       7             2               0.22
Logistics & Warehouse Jobs                      3             2               0.40
Creative & Design Jobs                          2

In [5]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np
def trunc(s, n=72):
    return " ".join(str(s).split()[:n])

core["d_full"] = core["description_clean"].fillna("")
core["d_72"] = core["d_full"].map(trunc)            # historical ko Adzuna jitna chhota karo
test = test.copy()
test["d_full"] = test["description_clean"].fillna("")
test["d_72"] = test["description_clean"].fillna("").map(trunc)

def fit_predict(train_df, test_df, col, weights=None):
    tf = TfidfVectorizer(max_features=20000, min_df=5, sublinear_tf=True, ngram_range=(1, 2), stop_words="english")
    Xa, Xb = tf.fit_transform(train_df[col]), tf.transform(test_df[col])
    m = LogisticRegression(max_iter=1000).fit(Xa, train_df["family"], sample_weight=weights)
    return m.predict(Xb)

def report(name, y, p):
    print(f"{name:52s} top1 {(p == y).mean():.3f}  macroF1 {f1_score(y, p, average='macro'):.3f}")

y = test["family"]
pA = fit_predict(core, test, "d_full"); report("A) hist (full text) -> live", y, pA)
pB = fit_predict(core, test, "d_72");   report("B) hist truncated to 72 words -> live", y, pB)

print("\nBiggest confusions in B (true -> predicted):")
cf = pd.crosstab(y.values, pB, rownames=["true"], colnames=["pred"])
pairs = [(t, p, int(cf.loc[t, p])) for t in cf.index for p in cf.columns if t != p]
print(pd.DataFrame(sorted(pairs, key=lambda x: -x[2])[:10], columns=["true", "pred", "n"]).to_string(index=False))

# C) kya live data ka ek hissa training mein daalne se fayda hota hai? (company-wise split, leakage nahi)
gss = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=0)
tr_i, te_i = next(gss.split(test, groups=test["companyName"].fillna("unk")))
live_tr, live_te = test.iloc[tr_i], test.iloc[te_i]
print(f"\nlive split by company: train {len(live_tr)} | test {len(live_te)}")
p0 = fit_predict(core, live_te, "d_72"); report("C0) hist only -> live_test", live_te["family"], p0)
for w in (1, 3):
    both = pd.concat([core[["d_72", "family"]], live_tr[["d_72", "family"]]], ignore_index=True)
    wts = np.r_[np.ones(len(core)), np.full(len(live_tr), float(w))]
    pc = fit_predict(both, live_te, "d_72", wts)
    report(f"C{w}) hist + live_train (live weight {w}) -> live_test", live_te["family"], pc)

A) hist (full text) -> live                          top1 0.604  macroF1 0.499
B) hist truncated to 72 words -> live                top1 0.610  macroF1 0.503

Biggest confusions in B (true -> predicted):
                  true               pred  n
        DevOps / Cloud Software Developer 89
  ERP / CRM Consultant Software Developer 89
Data Scientist / ML-AI Software Developer 71
         Data Engineer Software Developer 58
          QA / Testing Software Developer 53
     Data / BI Analyst Software Developer 30
      Mobile Developer Software Developer 29
Tech Project / Product Software Developer 27
        Database Admin Software Developer 26
         Cybersecurity Software Developer 23

live split by company: train 797 | test 1114
C0) hist only -> live_test                           top1 0.596  macroF1 0.462
C1) hist + live_train (live weight 1) -> live_test   top1 0.608  macroF1 0.487
C3) hist + live_train (live weight 3) -> live_test   top1 0.669  macroF1 0.537


In [6]:
from sklearn.model_selection import GroupShuffleSplit

def fit_proba(train_df, test_df, col):
    tf = TfidfVectorizer(max_features=20000, min_df=5, sublinear_tf=True, ngram_range=(1, 2), stop_words="english")
    Xa, Xb = tf.fit_transform(train_df[col]), tf.transform(test_df[col])
    m = LogisticRegression(max_iter=1000).fit(Xa, train_df["family"])
    return m.predict_proba(Xb), m.classes_

def alpha_table(P, cls, y_true, prior):
    rows = []
    for a in [0, 0.25, 0.5, 0.75, 1.0]:
        pred = cls[(P / (prior ** a)).argmax(1)]
        rows.append({"alpha": a, "top1": round((pred == y_true).mean(), 3),
                     "macroF1": round(f1_score(y_true, pred, average="macro"), 3),
                     "pred_as_SoftwareDev_%": round(100 * (pred == "Software Developer").mean(), 1)})
    return pd.DataFrame(rows)

# (1) Live data pe (2025 train -> live test)
P, cls = fit_proba(core, test, "d_72")
prior = core["family"].value_counts(normalize=True).reindex(cls).to_numpy()
print("LIVE test (true share of Software Developer: %.1f%%)" % (100 * (test["family"] == "Software Developer").mean()))
print(alpha_table(P, cls, test["family"].to_numpy(), prior).to_string(index=False))

# (2) Historical (in-domain): company-wise 80/20 split, taaki trade-off dikhe
g = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=1)
a_i, b_i = next(g.split(core, groups=core["companyName"].fillna("unk")))
ca, cb = core.iloc[a_i], core.iloc[b_i]
P2, cls2 = fit_proba(ca, cb, "d_72")
prior2 = ca["family"].value_counts(normalize=True).reindex(cls2).to_numpy()
print("\nHISTORICAL held-out companies (in-domain)")
print(alpha_table(P2, cls2, cb["family"].to_numpy(), prior2).to_string(index=False))

LIVE test (true share of Software Developer: 37.5%)
 alpha  top1  macroF1  pred_as_SoftwareDev_%
  0.00 0.610    0.503                   61.6
  0.25 0.623    0.547                   55.0
  0.50 0.621    0.563                   44.8
  0.75 0.587    0.534                   33.1
  1.00 0.529    0.488                   19.1

HISTORICAL held-out companies (in-domain)
 alpha  top1  macroF1  pred_as_SoftwareDev_%
  0.00 0.614    0.560                   49.7
  0.25 0.619    0.601                   43.4
  0.50 0.615    0.613                   37.5
  0.75 0.583    0.583                   30.0
  1.00 0.507    0.509                   19.6


In [7]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd
from pathlib import Path
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression

DRIVE_BASE = "/content/drive/MyDrive/Job recommendation nd skill gap analysis"
df = pd.read_pickle(Path(DRIVE_BASE) / "processed" / "jobs_checkpoint_v2.pkl")

OTHER = "Other / Non-IT or Unclear"; AMBIG = "IT Services - Ambiguous Role"
if "p_it_skills" not in df.columns:
    p = pd.read_pickle(Path(DRIVE_BASE) / "processed" / "p_it_skills_v1.pkl")
    df = df.merge(p, on="jobId", how="left")
real_fams = sorted(f for f in df["family"].unique() if f not in (OTHER, AMBIG))

# Family model: SIRF rule-labelled IT rows pe train
train = df[df["family"].isin(real_fams)]
cv_f = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3)
tf_f = TfidfVectorizer(max_features=20000, min_df=5, sublinear_tf=True, ngram_range=(1, 2), stop_words="english")
Xd = hstack([cv_f.fit_transform(train["skills_list"]), tf_f.fit_transform(train["description_clean"])]).tocsr()
model_fam = LogisticRegression(max_iter=1000).fit(Xd, train["family"])
ALPHA = 0.5        # prior correction (frozen): p / prior**0.5
prior_fam = train["family"].value_counts(normalize=True).reindex(model_fam.classes_).to_numpy()

def predict_family(frame):
    X = hstack([cv_f.transform(frame["skills_list"]), tf_f.transform(frame["description_clean"])]).tocsr()
    P = model_fam.predict_proba(X)
    adj = P / (prior_fam ** ALPHA)
    adj = adj / adj.sum(axis=1, keepdims=True)
    return model_fam.classes_[adj.argmax(1)], adj.max(1)

ext = df.copy()
ext["family_final"] = ext["family"]
ext["family_source"] = np.where(ext["family"].isin(real_fams), "rule",
                        np.where(ext["family"].eq(AMBIG), "ambiguous", "none"))
ext["pred_conf"] = np.nan
ext["it_tier"] = "non_IT_or_unknown"
ext.loc[ext["family"].isin(real_fams), "it_tier"] = "core_rule"
ext.loc[ext["family"].eq(AMBIG), "it_tier"] = "core_ambiguous"

cand = ext[ext["family"].eq(OTHER) & (ext["p_it_skills"] >= 0.15)]
fam_pred, conf = predict_family(cand)
ext.loc[cand.index, "family_final"] = fam_pred
ext.loc[cand.index, "family_source"] = "predicted"
ext.loc[cand.index, "pred_conf"] = conf
ext.loc[cand.index, "it_tier"] = np.where(cand["p_it_skills"] >= 0.5, "extended_high", "extended_mid")

print(ext["it_tier"].value_counts().to_string())
print("\nfamily_final counts (core + predicted):")
print(ext[ext["family_source"].isin(["rule", "predicted"])]["family_final"].value_counts().to_string())
ext.to_pickle(Path(DRIVE_BASE) / "processed" / "jobs_extended_v2.pkl")
print("\nsaved jobs_extended_v2.pkl:", ext.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
it_tier
non_IT_or_unknown    60119
core_rule            21401
extended_mid          6235
extended_high         4461
core_ambiguous        1513

family_final counts (core + predicted):
family_final
Software Developer            10295
ERP / CRM Consultant           5534
QA / Testing                   2313
IT Support / Sys & Network     1896
Data Engineer                  1888
DevOps / Cloud                 1715
Data Scientist / ML-AI         1592
Frontend / Web                 1170
Data / BI Analyst              1131
Business / Systems Analyst     1035
Tech Project / Product          958
Cybersecurity                   929
IT Architect                    894
Database Admin                  390
Mobile Developer                357

saved jobs_extended_v2.pkl: (93729, 33)
